<img src="https://www.efrei.fr/wp-content/uploads/2025/09/Logo-Efrei-CMJN-scaled.png"
     alt="EFREI logo" width="220">

# Access to General Practitioners in France

### Individual Student Project: Streamlit Dashboard

- **Author:** Yanis AMARA
- **Email:** [Yanis.amara@efrei.net](mailto:Yanis.amara@efrei.net)
- **Programme:** DATA ENGINEERING
- **Academic year:** 2026-2027
- **Course:** Data Analysis & Visualization
- **Professor:** Dr. Mano MATHEW
- **Institution:** EFREI
- **LinkedIn:** [Yanis AMARA](https://www.linkedin.com/in/yanis-amara-3a3953292/)

---

# Access to General Practitioners in France

## Project framing

**Motivation:** I will soon begin a research internship in healthcare.
I want to explore territorial inequalities in access to care in France
and better understand the concerns raised in public discussions.

**Target audience:** Municipalities, healthcare centres and public
health authorities seeking to understand local differences in access
to general practitioners.

**Main question:** How does potential access to general practitioners
differ across French territories in 2024?

**Supporting questions:**
1. Which departments have the highest and lowest potential access?
2. How much does access vary between communes within a department?
3. How did average access change between 2023 and 2024?

**Dataset:** DREES Localised Potential Accessibility (APL) to general
practitioners, referenced on data.gouv.fr.

**Scope:** France excluding Mayotte, focusing on 2024, with a
comparison to 2023 where geographical coverage is comparable.

**Hypothesis:** Departmental averages may conceal substantial
differences in potential access between communes.

**Expected takeaway:** Identify territories with lower potential
access, reveal inequalities within departments and describe
recent changes.

## 2. Data source and indicator

### Source

- **Producer:** DREES, using Cnam and Insee data.
- **Dataset:** Localised Potential Accessibility (APL)
  to healthcare professionals.
- **Portal:** [data.gouv.fr](https://www.data.gouv.fr/datasets/accessibilite-potentielle-localisee-apl-aux-professionnels-de-sante).
- **Licence:** Licence Ouverte / Open Licence version 2.0.
- **Study area:** France excluding Mayotte.
- **Main year:** 2024, with a comparison to 2023.

### What does APL measure?

APL estimates potential access to healthcare by considering:

- The activity of healthcare professionals.
- The population's healthcare needs according to age.
- The distance between residents and nearby healthcare providers.

For general practitioners, it is expressed as annual consultations,
visits or teleconsultations potentially accessible per
age-standardised inhabitant.

Our main analysis will use the indicator restricted to general
practitioners aged 65 or younger.

### Why this dataset?

Its local geographical detail supports comparisons between
departments and exploration of inequalities within departments.

### Interpretation limits

- APL measures potential access, not appointments actually obtained,
  waiting times or quality of care.

- Teleconsultations are included in the GP activity used to calculate
  APL. However, the dataset does not distinguish remote consultations
  from in-person care or identify booking platforms such as Doctolib.

- A geographically modelled indicator does not fully describe access
  to remote care, digital equipment or patients' digital skills.

- Teleconsultation may help some patients overcome geographical
  barriers. This dataset cannot establish who uses it or whether
  it compensates for limited local healthcare provision.

- Differences between territories do not establish their causes.

- Comparisons between years require consistent geographical coverage
  and attention to methodological limitations.

### Policy context

Context reviewed on 20 October 2026.

The discussion will consider the 2025 government pact against
medical deserts and the France Santé network, including its
2026 deployment and funding framework.

Policy objectives, implementation and demonstrated outcomes
must be distinguished.

Our 2023-2024 observations predate these initiatives and cannot
establish whether they have improved access. Assessing their
effects would require later observations and an appropriate
evaluation method.

In [4]:
from pathlib import Path
import requests

# Create a folder for the unchanged source data.
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASET_URL = (
    "https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/datasets/"
    "530_l-accessibilite-potentielle-localisee-apl/attachments/"
    "indicateur_d_apl_aux_medecins_generalistes_xlsx"
)

FILE_PATH = RAW_DIR / "apl_general_practitioners.xlsx"

# Download only if the file is not already available.
if not FILE_PATH.exists():
    response = requests.get(DATASET_URL, timeout=60)
    response.raise_for_status()
    FILE_PATH.write_bytes(response.content)
    print("Dataset downloaded.")
else:
    print("Using the existing dataset.")

print(f"File: {FILE_PATH}")
print(f"Size: {FILE_PATH.stat().st_size / 1_000_000:.2f} MB")

Dataset downloaded.
File: data/raw/apl_general_practitioners.xlsx
Size: 7.34 MB


In [5]:
import pandas as pd

# Inspect the workbook and its available sheets.
workbook = pd.ExcelFile(FILE_PATH, engine="openpyxl")
print("Available worksheets:", workbook.sheet_names)

# Preview the original layout before choosing the column headers.
preview = pd.read_excel(
    workbook,
    sheet_name="APL 2024",
    header=None,
    nrows=12
)

display(preview)

Available worksheets: ['Paramètres', 'APL 2022', 'APL 2023', 'APL 2024']


,0,1,2,3,4,5,6,7
0,Indicateur d'accessibilité potentielle localis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Millésime : 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sources : Activité des médecins généralistes l...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Champ : Omnipraticiens libéraux, certains méde...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Lecture : Compte tenu de la structure par âge ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Note : Pour calculer un APL moyen (agrégé au n...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Code commune INSEE,Commune,APL aux médecins généralistes,APL aux médecins généralistes de 65 ans ou moins,APL aux médecins généralistes de 62 ans ou moins,APL aux médecins généralistes de 60 ans ou moins,Population standardisée 2022 pour la médecine ...,Population totale 2022
9,NaN,NaN,En nombre de consultations/visites accessibles...,En nombre de consultations/visites accessibles...,En nombre de consultations/visites accessibles...,En nombre de consultations/visites accessibles...,En nombre d'habitants standardisés,En nombre d'habitants


In [6]:
df_2024_raw = pd.read_excel(
    "data/raw/apl_general_practitioners.xlsx",
    engine="openpyxl",
    sheet_name="APL 2024",
    header=8,
    skiprows=[9],
    usecols="A:H",
    dtype={"Code commune INSEE": "string"}
)

print(f"Rows: {df_2024_raw.shape[0]:,}")
print(f"Columns: {df_2024_raw.shape[1]}")

display(df_2024_raw.head())
df_2024_raw.info()

Rows: 34,890
Columns: 8


,Code commune INSEE,Commune,APL aux médecins généralistes,APL aux médecins généralistes de 65 ans ou moins,APL aux médecins généralistes de 62 ans ou moins,APL aux médecins généralistes de 60 ans ou moins,Population standardisée 2022 pour la médecine générale,Population totale 2022
0,01001,L'Abergement-Clémenciat,1.953,1.789,1.513,1.259,864.967,859.0
1,01002,L'Abergement-de-Varey,2.211,1.767,1.345,1.345,264.671,273.0
2,01004,Ambérieu-en-Bugey,3.098,2.520,2.029,1.907,15282.591,15554.0
3,01005,Ambérieux-en-Dombes,3.476,3.240,3.028,2.932,1884.510,1917.0
4,01006,Ambléon,0.953,0.847,0.645,0.531,119.188,114.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34890 entries, 0 to 34889
Data columns (total 8 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   Code commune INSEE                                      34890 non-null  string 
 1   Commune                                                 34890 non-null  object 
 2   APL aux médecins généralistes                           34890 non-null  float64
 3   APL aux médecins généralistes de 65 ans ou moins        34890 non-null  float64
 4   APL aux médecins généralistes de 62 ans ou moins        34890 non-null  float64
 5   APL aux médecins généralistes de 60 ans ou moins        34890 non-null  float64
 6   Population standardisée 2022 pour la médecine générale  34890 non-null  float64
 7   Population totale 2022                                  34890 non-null  float64
dtypes: float64(6), object(1), string(1)
m

In [7]:
code_column = "Code commune INSEE"

numeric_columns = df_2024_raw.select_dtypes(include="number").columns

# Check the format of geographical identifiers, including Corsica.
valid_codes = df_2024_raw[code_column].str.fullmatch(
    r"(?:\d{5}|2[AB]\d{3})",
    na=False
)

quality_checks = pd.Series(
    {
        "Missing cells": df_2024_raw.isna().sum().sum(),
        "Exact duplicate rows": df_2024_raw.duplicated().sum(),
        "Repeated commune codes": df_2024_raw[code_column].duplicated().sum(),
        "Invalid commune codes": (~valid_codes).sum(),
        "Negative numeric values": (
            df_2024_raw[numeric_columns] < 0
        ).sum().sum()
    },
    name="Count"
)

display(quality_checks.to_frame())

# Inspect observed ranges without removing any values.
display(df_2024_raw[numeric_columns].agg(["min", "max"]))

,Count
Missing cells,0
Exact duplicate rows,0
Repeated commune codes,0
Invalid commune codes,0
Negative numeric values,0


,APL aux médecins généralistes,APL aux médecins généralistes de 65 ans ou moins,APL aux médecins généralistes de 62 ans ou moins,APL aux médecins généralistes de 60 ans ou moins,Population standardisée 2022 pour la médecine générale,Population totale 2022
min,0.000,0.000,0.000,0.000,1.916,2.0
max,23.904,23.904,23.904,15.936,479250.450,511684.0


In [8]:
# Keep the imported table unchanged.
df_2024 = df_2024_raw.copy()

# Remove surrounding spaces from column headings.
df_2024.columns = df_2024.columns.str.strip()

df_2024 = df_2024.rename(
    columns={
        "Code commune INSEE": "commune_code",
        "Commune": "commune_name",
        "APL aux médecins généralistes": "apl_all_ages",
        "APL aux médecins généralistes de 65 ans ou moins": "apl_65_or_younger",
        "APL aux médecins généralistes de 62 ans ou moins": "apl_62_or_younger",
        "APL aux médecins généralistes de 60 ans ou moins": "apl_60_or_younger",
        "Population standardisée 2022 pour la médecine générale": "standardised_population",
        "Population totale 2022": "total_population"
    },
    errors="raise"
)

# Distinguish the indicator year from the population reference year.
df_2024["apl_year"] = 2024
df_2024["population_year"] = 2022

display(df_2024.head())
print(f"Prepared table: {df_2024.shape[0]:,} rows, {df_2024.shape[1]} columns")

,commune_code,commune_name,apl_all_ages,apl_65_or_younger,apl_62_or_younger,apl_60_or_younger,standardised_population,total_population,apl_year,population_year
0,01001,L'Abergement-Clémenciat,1.953,1.789,1.513,1.259,864.967,859.0,2024,2022
1,01002,L'Abergement-de-Varey,2.211,1.767,1.345,1.345,264.671,273.0,2024,2022
2,01004,Ambérieu-en-Bugey,3.098,2.520,2.029,1.907,15282.591,15554.0,2024,2022
3,01005,Ambérieux-en-Dombes,3.476,3.240,3.028,2.932,1884.510,1917.0,2024,2022
4,01006,Ambléon,0.953,0.847,0.645,0.531,119.188,114.0,2024,2022


Prepared table: 34,890 rows, 10 columns


In [9]:
APL_COLUMN = "apl_65_or_younger"

# Calculate the population-weighted APL.
weighted_apl_sum = (
    df_2024[APL_COLUMN] * df_2024["standardised_population"]
).sum()

total_weight = df_2024["standardised_population"].sum()
national_apl = weighted_apl_sum / total_weight

# Compare with an unweighted mean for learning purposes.
unweighted_apl = df_2024[APL_COLUMN].mean()
covered_population = df_2024["total_population"].sum()

print("Scope: France excluding Mayotte; GPs aged 65 or younger")
print(f"National APL in 2024 (weighted): {national_apl:.3f}")
print(f"Mean across geographical units (unweighted): {unweighted_apl:.3f}")
print(f"Covered population, reference year 2022: {covered_population:,.0f}")

Scope: France excluding Mayotte; GPs aged 65 or younger
National APL in 2024 (weighted): 3.259
Mean across geographical units (unweighted): 2.545
Covered population, reference year 2022: 67,760,573


In [10]:
# Derive department codes while preserving Corsica and overseas codes.
df_2024["department_code"] = df_2024["commune_code"].str[:2]

overseas = df_2024["commune_code"].str.startswith("97")
df_2024.loc[overseas, "department_code"] = (
    df_2024.loc[overseas, "commune_code"].str[:3]
)

# Create a temporary table containing the weighting calculation.
weighted_data = df_2024.assign(
    weighted_apl=df_2024[APL_COLUMN] * df_2024["standardised_population"]
)

department_summary = weighted_data.groupby(
    "department_code", as_index=False
).agg(
    weighted_apl_sum=("weighted_apl", "sum"),
    standardised_population=("standardised_population", "sum"),
    total_population=("total_population", "sum"),
    geographical_units=("commune_code", "nunique")
)

department_summary["apl_65_or_younger"] = (
    department_summary["weighted_apl_sum"]
    / department_summary["standardised_population"]
)

department_summary = department_summary.sort_values("apl_65_or_younger")

print(f"Departments covered: {len(department_summary)}")

display(
    department_summary[
        ["department_code", "apl_65_or_younger", "total_population"]
    ].head(10)
)

Departments covered: 100


,department_code,apl_65_or_younger,total_population
98,973,1.413769,288382.0
77,77,1.944510,1452399.0
26,28,1.950573,432950.0
91,91,1.957733,1324546.0
78,78,1.990656,1470778.0
17,18,1.995454,299496.0
95,95,2.063106,1270845.0
89,89,2.094893,333896.0
45,45,2.137600,687063.0
58,58,2.163151,202299.0


In [11]:
# Retrieve official department codes and names.
response = requests.get(
    "https://geo.api.gouv.fr/departements",
    timeout=60
)
response.raise_for_status()

department_reference = pd.DataFrame(response.json())[
    ["code", "nom"]
].rename(
    columns={
        "code": "department_code",
        "nom": "department_name"
    }
)

department_reference["department_code"] = (
    department_reference["department_code"].astype("string")
)

# Add names while preserving every department in our analysis.
department_analysis = department_summary.merge(
    department_reference,
    on="department_code",
    how="left",
    validate="one_to_one"
)

unmatched = department_analysis["department_name"].isna().sum()

print(f"Departments after joining: {len(department_analysis)}")
print(f"Departments without a matching name: {unmatched}")

display(
    department_analysis[
        ["department_code", "department_name",
         "apl_65_or_younger", "total_population"]
    ].head(10)
)

Departments after joining: 100
Departments without a matching name: 0


,department_code,department_name,apl_65_or_younger,total_population
0,973,Guyane,1.413769,288382.0
1,77,Seine-et-Marne,1.944510,1452399.0
2,28,Eure-et-Loir,1.950573,432950.0
3,91,Essonne,1.957733,1324546.0
4,78,Yvelines,1.990656,1470778.0
5,18,Cher,1.995454,299496.0
6,95,Val-d'Oise,2.063106,1270845.0
7,89,Yonne,2.094893,333896.0
8,45,Loiret,2.137600,687063.0
9,58,Nièvre,2.163151,202299.0


In [12]:
import plotly.express as px

lowest_departments = department_analysis.nsmallest(10, APL_COLUMN)

fig_rank = px.bar(
    lowest_departments,
    x=APL_COLUMN,
    y="department_name",
    orientation="h",
    text_auto=".2f",
    hover_data={
        APL_COLUMN: ":.3f",
        "department_code": True,
        "total_population": ":,.0f"
    },
    labels={
        APL_COLUMN: "APL (consultations/year/standardised inhabitant)",
        "department_name": "Department",
        "department_code": "Department code",
        "total_population": "Population (2022)"
    },
    title=(
        "Ten departments with the lowest potential GP access, 2024"
        "<br><sup>GPs aged 65 or younger; France excluding Mayotte. "
        "Source: DREES.</sup>"
    ),
    color_discrete_sequence=["#245A81"],
    range_x=[0, national_apl * 1.25],
    template="plotly_white",
    height=550
)

# Place the lowest value at the top.
fig_rank.update_yaxes(autorange="reversed")

# Compare departmental values with the population-weighted national mean.
fig_rank.add_vline(
    x=national_apl,
    line_dash="dash",
    line_color="#9C2E3C",
    annotation_text=f"National APL: {national_apl:.2f}",
    annotation_position="top right"
)

fig_rank.show()

In [13]:
import json
from pathlib import Path
import requests

GEOJSON_URL = (
    "https://etalab-datasets.geo.data.gouv.fr/"
    "2024/geojson/departements-1000m.geojson"
)

ASSETS_DIR = Path("/content/assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
GEOJSON_PATH = ASSETS_DIR / "departments_2024.geojson"

if not GEOJSON_PATH.exists():
    response = requests.get(GEOJSON_URL, timeout=60)
    response.raise_for_status()
    GEOJSON_PATH.write_bytes(response.content)

departments_geojson = json.loads(
    GEOJSON_PATH.read_text(encoding="utf-8")
)

# Check that every analysed department has a geographical boundary.
features = departments_geojson["features"]

boundary_codes = {
    feature["properties"]["code"]
    for feature in features
    if feature.get("geometry")
}

analysis_codes = set(department_analysis["department_code"])

print("Boundary features:", len(features))
print("Analysed departments without boundaries:",
      sorted(analysis_codes - boundary_codes))
print("Boundary codes without APL observations:",
      sorted(boundary_codes - analysis_codes))

Boundary features: 109
Analysed departments without boundaries: []
Boundary codes without APL observations: ['975', '976', '977', '978', '984', '986', '987', '988', '989']


In [14]:
MAP_VIEW = "Metropolitan France"
MAP_VIEW = "Guyane"

map_views = {
    "Metropolitan France": department_analysis.loc[
        ~department_analysis["department_code"].str.startswith("97"),
        "department_code"
    ].tolist(),
    "Guadeloupe": ["971"],
    "Martinique": ["972"],
    "Guyane": ["973"],
    "La Réunion": ["974"]
}

map_data = department_analysis.loc[
    department_analysis["department_code"].isin(map_views[MAP_VIEW])
]

map_codes = set(map_data["department_code"])

map_geojson = {
    "type": "FeatureCollection",
    "features": [
        feature for feature in departments_geojson["features"]
        if feature["properties"]["code"] in map_codes
    ]
}

fig_map = px.choropleth(
    map_data,
    geojson=map_geojson,
    locations="department_code",
    featureidkey="properties.code",
    color=APL_COLUMN,
    hover_name="department_name",
    hover_data={
        APL_COLUMN: ":.3f",
        "department_code": True,
        "total_population": ":,.0f"
    },
    labels={
        APL_COLUMN: "APL",
        "department_code": "Department code",
        "total_population": "Population (2022)"
    },
    color_continuous_scale="Cividis",
    range_color=[
        department_analysis[APL_COLUMN].min(),
        department_analysis[APL_COLUMN].max()
    ],
    projection="mercator",
    title=(
        f"Potential GP access in 2024: {MAP_VIEW}"
        "<br><sup>GPs aged 65 or younger. "
        "Source: DREES; boundaries: Etalab/IGN.</sup>"
    ),
    template="plotly_white",
    height=600
)

fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_traces(marker_line_color="white", marker_line_width=0.5)
fig_map.update_layout(
    coloraxis_colorbar={
        "title": "Consultations/year<br>per standardised<br>inhabitant"
    }
)

print(f"Map view: {MAP_VIEW}; departments displayed: {len(map_data)}")
fig_map.show()

Map view: Guyane; departments displayed: 1


In [15]:
SELECTED_DEPARTMENT = "973"

# Select communes and retrieve the department's weighted APL.
selected_units = df_2024.loc[
    df_2024["department_code"] == SELECTED_DEPARTMENT
].copy()

department_row = department_analysis.loc[
    department_analysis["department_code"] == SELECTED_DEPARTMENT
].iloc[0]

department_name = department_row["department_name"]
department_apl = department_row[APL_COLUMN]

# Group communes by APL and sum their resident populations.
fig_distribution = px.histogram(
    selected_units,
    x=APL_COLUMN,
    y="total_population",
    histfunc="sum",
    histnorm="percent",
    nbins=15,
    labels={
        APL_COLUMN: "APL (consultations/year/standardised inhabitant)",
        "total_population": "Residents (2022)",
    },
    title=(
        f"Beyond the average: access within {department_name}, 2024"
        "<br><sup>GPs aged 65 or younger. Source: DREES.</sup>"
    ),
    color_discrete_sequence=["#245A81"],
    template="plotly_white",
    height=500,
)

fig_distribution.update_layout(bargap=0.08)
fig_distribution.update_yaxes(
    title_text="Share of department population (%)",
    ticksuffix="%",
)

fig_distribution.add_vline(
    x=department_apl,
    line_dash="dash",
    line_color="#9C2E3C",
    annotation_text=f"Department APL: {department_apl:.2f}",
    annotation_position="top right",
)

print(f"Department: {department_name}")
print(f"Geographical units: {len(selected_units):,}")
print(f"Population (2022): {selected_units['total_population'].sum():,.0f}")

fig_distribution.show()

Department: Guyane
Geographical units: 22
Population (2022): 288,382


In [16]:
department_population = selected_units["total_population"].sum()

# Identify communes with zero APL for the selected GP age group.
zero_apl_units = selected_units.loc[
    selected_units[APL_COLUMN].eq(0)
].copy()

zero_apl_population = zero_apl_units["total_population"].sum()

# Count residents living in communes below the national benchmark.
below_national_population = selected_units.loc[
    selected_units[APL_COLUMN].lt(national_apl),
    "total_population",
].sum()

print(f"Communes with zero APL: {len(zero_apl_units)}")
print(f"Residents in these communes (2022): {zero_apl_population:,.0f}")
print(
    f"Share of department population: "
    f"{100 * zero_apl_population / department_population:.1f}%"
)
print(
    f"Population living in communes below national APL: "
    f"{100 * below_national_population / department_population:.1f}%"
)

# Compare the selected indicator with access to GPs of all ages.
display(
    zero_apl_units[
        ["commune_name", APL_COLUMN, "apl_all_ages", "total_population"]
    ]
    .sort_values("total_population", ascending=False)
    .reset_index(drop=True)
)

Communes with zero APL: 12
Residents in these communes (2022): 47,077
Share of department population: 16.3%
Population living in communes below national APL: 100.0%


,commune_name,apl_65_or_younger,apl_all_ages,total_population
0,Apatou,0.0,0.421,10059.0
1,Grand-Santi,0.0,0.000,9390.0
2,Maripasoula,0.0,0.000,8423.0
3,Papaichton,0.0,0.000,5530.0
4,Saint-Georges,0.0,1.320,4710.0
5,Sinnamary,0.0,0.000,2801.0
6,Camopi,0.0,0.000,2168.0
7,Iracoubo,0.0,0.000,1685.0
8,Régina,0.0,0.000,1657.0
9,Saül,0.0,0.000,297.0


In [17]:
# Load the previous year using the same workbook structure.
df_2023 = pd.read_excel(
    FILE_PATH,
    engine="openpyxl",
    sheet_name="APL 2023",
    header=8,
    skiprows=[9],
    usecols="A:H",
    dtype={"Code commune INSEE": "string"},
)

df_2023.columns = df_2023.columns.str.strip()

# Match the English column names used for 2024.
df_2023 = df_2023.rename(
    columns={
        "Code commune INSEE": "commune_code",
        "Commune": "commune_name",
        "APL aux médecins généralistes": "apl_all_ages",
        "APL aux médecins généralistes de 65 ans et moins": "apl_65_or_younger",
        "APL aux médecins généralistes de 62 ans et moins": "apl_62_or_younger",
        "APL aux médecins généralistes de 60 ans et moins": "apl_60_or_younger",
        "Population standardisée 2021 pour la médecine générale": "standardised_population",
        "Population totale 2021": "total_population",
    },
    errors="raise",
)

df_2023["apl_year"] = 2023
df_2023["population_year"] = 2021

print(f"Rows: {len(df_2023):,}")
print(f"Missing values: {df_2023.isna().sum().sum():,}")
print(f"Duplicate commune codes: {df_2023['commune_code'].duplicated().sum():,}")

display(df_2023.head(3))

Rows: 34,954
Missing values: 0
Duplicate commune codes: 0


,commune_code,commune_name,apl_all_ages,apl_65_or_younger,apl_62_or_younger,apl_60_or_younger,standardised_population,total_population,apl_year,population_year
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832,2023,2021
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267,2023,2021
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854,2023,2021


In [18]:
codes_2023 = set(df_2023["commune_code"])
codes_2024 = set(df_2024["commune_code"])

common_codes = codes_2023 & codes_2024
codes_only_2023 = codes_2023 - codes_2024
codes_only_2024 = codes_2024 - codes_2023

print(f"Codes present in both years: {len(common_codes):,}")
print(f"Codes present only in 2023: {len(codes_only_2023):,}")
print(f"Codes present only in 2024: {len(codes_only_2024):,}")

# Inspect examples of units that cannot be matched by code.
audit_columns = ["commune_code", "commune_name", "total_population"]

print("\nExamples: codes present only in 2023")
display(
    df_2023.loc[
        df_2023["commune_code"].isin(codes_only_2023),
        audit_columns,
    ].head(5)
)

print("\nExamples: codes present only in 2024")
display(
    df_2024.loc[
        df_2024["commune_code"].isin(codes_only_2024),
        audit_columns,
    ].head(5)
)

Codes present in both years: 34,886
Codes present only in 2023: 68
Codes present only in 2024: 4

Examples: codes present only in 2023


,commune_code,commune_name,total_population
277,01330,Ruffieu,177
690,02311,Filain,119
2623,08300,Le Mont-Dieu,12
3088,09287,Senconac,8
4074,12076,Conques-en-Rouergue,1573



Examples: codes present only in 2024


,commune_code,commune_name,total_population
4201,12218,Conques-en-Rouergue,1555.0
4820,14581,Aurseulles,1908.0
17548,49126,Orée d'Anjou,16975.0
27035,69114,Porte des Pierres Dorées,4079.0


In [19]:
# Reuse the same aggregation method for both years.
def calculate_weighted_apl(data):
    weights = data["standardised_population"]
    return (data[APL_COLUMN] * weights).sum() / weights.sum()


national_trend = pd.DataFrame([
    {
        "apl_year": 2023,
        "population_year": 2021,
        "weighted_apl": calculate_weighted_apl(df_2023),
    },
    {
        "apl_year": 2024,
        "population_year": 2022,
        "weighted_apl": calculate_weighted_apl(df_2024),
    },
])

# Calculate changes from the previous observation.
national_trend["absolute_change"] = national_trend["weighted_apl"].diff()
national_trend["relative_change_pct"] = (
    national_trend["weighted_apl"].pct_change() * 100
)

display(
    national_trend.round({
        "weighted_apl": 3,
        "absolute_change": 3,
        "relative_change_pct": 2,
    })
)

,apl_year,population_year,weighted_apl,absolute_change,relative_change_pct
0,2023,2021,3.295,NaN,NaN
1,2024,2022,3.259,-0.036,-1.09


In [20]:
# Identify departments, including three-digit overseas codes.
df_2023["department_code"] = df_2023["commune_code"].str[:2]

overseas_2023 = df_2023["commune_code"].str.startswith("97")
df_2023.loc[overseas_2023, "department_code"] = (
    df_2023.loc[overseas_2023, "commune_code"].str[:3]
)

# Reuse our weighted APL function for each department.
department_2023 = (
    df_2023.groupby("department_code")[
        [APL_COLUMN, "standardised_population"]
    ]
    .apply(calculate_weighted_apl)
    .rename("apl_2023")
    .reset_index()
)

# Match departmental results across years.
department_trend = (
    department_analysis[
        ["department_code", "department_name", APL_COLUMN]
    ]
    .rename(columns={APL_COLUMN: "apl_2024"})
    .merge(
        department_2023,
        on="department_code",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

assert department_trend["_merge"].eq("both").all(), (
    "Some departments do not match between years."
)
department_trend = department_trend.drop(columns="_merge")

department_trend["absolute_change"] = (
    department_trend["apl_2024"] - department_trend["apl_2023"]
)
department_trend["relative_change_pct"] = (
    department_trend["absolute_change"]
    / department_trend["apl_2023"] * 100
)

print(f"Departments matched: {len(department_trend)}")
print(f"APL increased: {(department_trend['absolute_change'] > 0).sum()}")
print(f"APL decreased: {(department_trend['absolute_change'] < 0).sum()}")

display(
    department_trend.nsmallest(5, "relative_change_pct").round(3)
)

Departments matched: 100
APL increased: 21
APL decreased: 79


,department_code,department_name,apl_2024,apl_2023,absolute_change,relative_change_pct
48,48,Lozère,2.226,2.447,-0.221,-9.031
82,82,Tarn-et-Garonne,2.669,2.932,-0.263,-8.985
98,973,Guyane,1.414,1.551,-0.138,-8.873
38,38,Isère,3.252,3.515,-0.264,-7.500
66,66,Pyrénées-Orientales,3.213,3.436,-0.223,-6.491


In [21]:
change_plot = department_trend.copy()

change_plot["direction"] = "Unchanged"
change_plot.loc[change_plot["absolute_change"] < 0, "direction"] = "Decrease"
change_plot.loc[change_plot["absolute_change"] > 0, "direction"] = "Increase"

fig_change = px.scatter(
    change_plot,
    x="apl_2024",
    y="relative_change_pct",
    color="direction",
    hover_name="department_name",
    hover_data={
        "department_code": True,
        "apl_2023": ":.3f",
        "apl_2024": ":.3f",
        "relative_change_pct": ":+.2f",
    },
    labels={
        "apl_2024": "2024 APL (consultations/year/standardised inhabitant)",
        "apl_2023": "2023 APL",
        "relative_change_pct": "Change from 2023 (%)",
        "department_code": "Department code",
        "direction": "Change",
    },
    color_discrete_map={
        "Decrease": "#B45F06",
        "Increase": "#2166AC",
        "Unchanged": "#777777",
    },
    title=(
        "Potential GP access: current level and recent change"
        "<br><sup>GPs aged 65 or younger; France excluding Mayotte. "
        "Source: DREES.</sup>"
    ),
    template="plotly_white",
    height=600,
)

fig_change.update_traces(marker_size=10, marker_opacity=0.8)
fig_change.update_yaxes(ticksuffix="%")

fig_change.add_hline(
    y=0, line_dash="dash", line_color="#666666"
)
fig_change.add_vline(
    x=national_apl,
    line_dash="dot",
    line_color="#666666",
    annotation_text=f"National APL: {national_apl:.2f}",
    annotation_position="top right",
)

fig_change.show()

In [22]:
# Select departments meeting both conditions.
below_and_declining = department_trend.loc[
    (department_trend["apl_2024"] < national_apl)
    & (department_trend["absolute_change"] < 0)
].copy()

# Add their 2022 resident populations.
below_and_declining = below_and_declining.merge(
    department_analysis[["department_code", "total_population"]],
    on="department_code",
    how="left",
    validate="one_to_one",
)

selected_population = below_and_declining["total_population"].sum()
national_population = department_analysis["total_population"].sum()
population_share = 100 * selected_population / national_population

print(f"Departments meeting both conditions: {len(below_and_declining)}")
print(f"Residents in these departments (2022): {selected_population:,.0f}")
print(f"Share of covered population: {population_share:.1f}%")

display(
    below_and_declining.nsmallest(5, "apl_2024")[
        [
            "department_name",
            "apl_2024",
            "relative_change_pct",
            "total_population",
        ]
    ].round(3)
)

Departments meeting both conditions: 50
Residents in these departments (2022): 25,788,500
Share of covered population: 38.1%


,department_name,apl_2024,relative_change_pct,total_population
49,Guyane,1.414,-8.873,288382.0
37,Seine-et-Marne,1.945,-4.056,1452399.0
15,Eure-et-Loir,1.951,-5.359,432950.0
47,Essonne,1.958,-6.107,1324546.0
38,Yvelines,1.991,-5.134,1470778.0


In [23]:
import numpy as np

apl_columns = [
    "apl_all_ages",
    "apl_65_or_younger",
    "apl_62_or_younger",
    "apl_60_or_younger",
]
numeric_columns = apl_columns + [
    "standardised_population",
    "total_population",
]


def audit_numeric_rules(data):
    numeric = data[numeric_columns]

    age_order_errors = (
        data[apl_columns].diff(axis=1).gt(1e-9).any(axis=1)
    )
    fractional_populations = (
        data["total_population"]
        .sub(data["total_population"].round())
        .abs()
        .gt(1e-6)
    )

    return pd.Series({
        "Non-finite numeric cells": (~np.isfinite(numeric.to_numpy())).sum(),
        "Negative numeric cells": numeric.lt(0).sum().sum(),
        "Age-subset order violations": age_order_errors.sum(),
        "Non-integer resident populations": fractional_populations.sum(),
        "Zero standardised populations": data["standardised_population"].eq(0).sum(),
        "Zero resident populations": data["total_population"].eq(0).sum(),
    })


numeric_audit = pd.DataFrame({
    2023: audit_numeric_rules(df_2023),
    2024: audit_numeric_rules(df_2024),
})
display(numeric_audit)

# Inspect observations with either population measure equal to zero.
display(
    df_2023.loc[
        df_2023[["standardised_population", "total_population"]]
        .eq(0).any(axis=1),
        [
            "commune_code",
            "commune_name",
            "standardised_population",
            "total_population",
        ],
    ]
)

,2023,2024
Non-finite numeric cells,0,0
Negative numeric cells,0,0
Age-subset order violations,0,0
Non-integer resident populations,0,0
Zero standardised populations,3,0
Zero resident populations,3,0


,commune_code,commune_name,standardised_population,total_population
23110,60694,Villotran,0.0,0
32736,85165,Oie,0.0,0
32775,85212,Sainte-Florence,0.0,0


In [24]:
# Inspect the five highest communal APL values.
highest_apl = df_2024.nlargest(5, APL_COLUMN)

display(
    highest_apl[
        [
            "commune_code",
            "commune_name",
            APL_COLUMN,
            "standardised_population",
            "total_population",
        ]
    ].round(3)
)

top5_weight_share = (
    100 * highest_apl["standardised_population"].sum()
    / df_2024["standardised_population"].sum()
)

# Create a temporary subset for a sensitivity check.
sensitivity_data = df_2024.loc[
    ~df_2024["commune_code"].isin(highest_apl["commune_code"])
]
apl_without_top5 = calculate_weighted_apl(sensitivity_data)

print(f"Top five: share of national aggregation weight: {top5_weight_share:.4f}%")
print(f"National APL, full dataset: {national_apl:.6f}")
print(f"National APL, excluding these five: {apl_without_top5:.6f}")

,commune_code,commune_name,apl_65_or_younger,standardised_population,total_population
1775,05085,Montgenèvre,23.904,451.817,459.0
1857,05177,Vars,18.049,598.379,592.0
20548,56086,Île-d'Houat,14.737,244.282,214.0
1983,06119,Saint-Dalmas-le-Selvage,12.729,109.339,102.0
1984,06120,Saint-Étienne-de-Tinée,12.729,1343.067,1269.0


Top five: share of national aggregation weight: 0.0041%
National APL, full dataset: 3.258882
National APL, excluding these five: 3.258369


In [25]:
PROJECT_DIR = Path("/content/apl_dashboard")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Combine both years and flag zero-population observations.
communes_export = pd.concat(
    [df_2023, df_2024],
    ignore_index=True,
)

communes_export["zero_population_flag"] = (
    communes_export[["standardised_population", "total_population"]]
    .eq(0)
    .any(axis=1)
)

tables_to_export = {
    "apl_communes_2023_2024.csv": communes_export,
    "apl_departments_2024.csv": department_analysis,
    "apl_department_trends.csv": department_trend,
    "apl_national_trend.csv": national_trend,
    "numeric_audit.csv": numeric_audit.rename_axis("check").reset_index(),
}

for filename, table in tables_to_export.items():
    table.to_csv(
        PROCESSED_DIR / filename,
        index=False,
        encoding="utf-8",
    )
    print(f"{filename}: {len(table):,} rows")

preparation_notes = """# Data preparation decisions

- Source: DREES Localised Potential Accessibility (APL).
- Scope: France excluding Mayotte, 2023 and 2024.
- Main indicator: access to GPs aged 65 or younger.
- Population reference years: 2021 for APL 2023; 2022 for APL 2024.
- Preserve commune codes as strings when loading CSV files.
- Retain zero APL values.
- Retain and flag the three zero-population observations in 2023.
  Their zero standardised weights contribute nothing to weighted averages.
- Retain high APL values. Excluding the five highest values changes
  national APL from 3.258882 to 3.258369.
- Use standardised population to aggregate APL.
- Use resident population for population counts and shares.
- Aggregate all observations for national and departmental comparisons.
  Do not interpret unmatched commune codes as changes in GP supply.
- Missing first-year changes in the national table indicate the baseline.
- The national benchmark is not an official shortage threshold.
"""

(PROJECT_DIR / "DATA_PREPARATION.md").write_text(
    preparation_notes,
    encoding="utf-8",
)

print("DATA_PREPARATION.md saved.")
print(f"Project folder: {PROJECT_DIR}")


apl_communes_2023_2024.csv: 69,844 rows
apl_departments_2024.csv: 100 rows
apl_department_trends.csv: 100 rows
apl_national_trend.csv: 2 rows
numeric_audit.csv: 6 rows
DATA_PREPARATION.md saved.
Project folder: /content/apl_dashboard


In [26]:
%pip install -q "streamlit==1.66.0"

import sys
from importlib.metadata import version

# Record the installed versions of the project dependencies.
packages = [
    "streamlit",
    "pandas",
    "numpy",
    "plotly",
    "openpyxl",
    "requests",
]

requirements_text = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
)

requirements_path = PROJECT_DIR / "requirements.txt"
requirements_path.write_text(
    requirements_text + "\n",
    encoding="utf-8",
)

print(f"Python: {sys.version.split()[0]}")
print(f"Saved: {requirements_path}\n")
print(requirements_text)

Python: 3.13.16
Saved: /content/apl_dashboard/requirements.txt

streamlit==1.66.0
pandas==2.2.3
numpy==2.1.3
plotly==5.24.1
openpyxl==3.1.5
requests==2.32.4


In [27]:
# Save the official EFREI logo inside the project.
APP_ASSETS_DIR = PROJECT_DIR / "assets"
APP_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

LOGO_PATH = APP_ASSETS_DIR / "efrei_logo.png"
LOGO_URL = (
    "https://www.efrei.fr/wp-content/uploads/2025/09/"
    "Logo-Efrei-CMJN-scaled.png"
)

if not LOGO_PATH.exists():
    response = requests.get(LOGO_URL, timeout=60)
    response.raise_for_status()
    LOGO_PATH.write_bytes(response.content)

# Create the first version of the Streamlit app.
APP_PATH = PROJECT_DIR / "app.py"

app_code = '''from pathlib import Path
import streamlit as st

BASE_DIR = Path(__file__).resolve().parent

st.set_page_config(
    page_title="GP Access in France",
    page_icon="🩺",
    layout="wide",
    initial_sidebar_state="expanded",
)

with st.sidebar:
    st.image(
        str(BASE_DIR / "assets" / "efrei_logo.png"),
        width=180,
        alt="EFREI logo",
    )
    st.markdown("**Yanis AMARA**")
    st.caption("DATA ENGINEERING · 2026–2027")

    with st.expander("Course and contact"):
        st.markdown("""
**Institution:** EFREI
**Course:** Data Analysis & Visualization
**Professor:** Dr. Mano MATHEW

[Yanis.amara@efrei.net](mailto:Yanis.amara@efrei.net)
[LinkedIn profile](https://www.linkedin.com/in/yanis-amara-3a3953292/)
""")

st.title("Access to General Practitioners in France")

st.write(
    "A national average can hide local gaps. Explore differences "
    "between departments, what their averages hide, and how potential "
    "access changed between 2023 and 2024."
)

st.caption(
    "Source: DREES APL · France excluding Mayotte · "
    "2024 snapshot with a 2023 comparison"
)

st.info(
    "APL estimates potential access in annual consultations per "
    "age-standardised inhabitant. This dashboard focuses on GPs "
    "aged 65 or younger."
)
'''

compile(app_code, str(APP_PATH), "exec")
APP_PATH.write_text(app_code, encoding="utf-8")

print(f"Logo saved: {LOGO_PATH}")
print(f"App created: {APP_PATH}")
print("Python syntax check passed.")

Logo saved: /content/apl_dashboard/assets/efrei_logo.png
App created: /content/apl_dashboard/app.py
Python syntax check passed.


In [28]:
filters_code = '''import pandas as pd
import streamlit as st


@st.cache_data(show_spinner=False)
def load_tables(data_dir):
    files = {
        "communes": "apl_communes_2023_2024.csv",
        "departments": "apl_departments_2024.csv",
        "trends": "apl_department_trends.csv",
        "national": "apl_national_trend.csv",
    }
    return {
        name: pd.read_csv(
            data_dir / filename,
            dtype={"commune_code": "string", "department_code": "string"},
        )
        for name, filename in files.items()
    }


def prepare_view(base_dir):
    tables = load_tables(base_dir / "data" / "processed")

    department_names = (
        tables["departments"]
        .set_index("department_code")["department_name"]
        .to_dict()
    )
    all_codes = sorted(department_names, key=department_names.get)

    area_codes = {
        "France excluding Mayotte": all_codes,
        "Metropolitan France": [
            code for code in all_codes if not code.startswith("97")
        ],
        "Guadeloupe": ["971"],
        "Martinique": ["972"],
        "Guyane": ["973"],
        "La Réunion": ["974"],
    }

    st.sidebar.divider()
    st.sidebar.subheader("Explore")

    area = st.sidebar.selectbox(
        "Area",
        options=list(area_codes),
        key="area",
    )
    selected = st.sidebar.multiselect(
        "Departments",
        options=area_codes[area],
        format_func=lambda code: f"{department_names[code]} ({code})",
        placeholder="All departments in this area",
        help="Leave empty to include the whole selected area.",
        key=f"departments_{area}",
    )
    selected_codes = selected or area_codes[area]

    view = {}
    for name in ("communes", "departments", "trends"):
        data = tables[name]
        view[name] = data.loc[
            data["department_code"].isin(selected_codes)
        ].copy()

    view["national"] = tables["national"]
    view["area"] = area
    return view
'''

module_path = PROJECT_DIR / "data_and_filters.py"
compile(filters_code, str(module_path), "exec")
module_path.write_text(filters_code, encoding="utf-8")

# Connect the module once, including when this cell is rerun.
app_text = APP_PATH.read_text(encoding="utf-8")
import_line = "from data_and_filters import prepare_view"
call_line = "view = prepare_view(BASE_DIR)"

if import_line not in app_text:
    app_text = app_text.replace(
        "import streamlit as st",
        f"import streamlit as st\n{import_line}",
        1,
    )

if call_line not in app_text:
    app_text += f"\n\n{call_line}\n"

compile(app_text, str(APP_PATH), "exec")
APP_PATH.write_text(app_text, encoding="utf-8")

# Run the app through Streamlit's testing interface.
from streamlit.testing.v1 import AppTest

test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."

print("App execution check passed.")
print("Default area:", test_app.sidebar.selectbox[0].value)
print(
    "Departments available:",
    len(test_app.sidebar.multiselect[0].options),
)

2026-10-24 09:48:22.581 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App execution check passed.
Default area: France excluding Mayotte
Departments available: 100


In [29]:
metrics_code = '''import streamlit as st


def weighted_apl(data):
    weights = data["standardised_population"]
    total_weight = weights.sum()
    if total_weight <= 0:
        return None
    return (data["apl_65_or_younger"] * weights).sum() / total_weight


def render_metrics(view):
    data = view["communes"]
    current = data.loc[data["apl_year"].eq(2024)]
    previous = data.loc[data["apl_year"].eq(2023)]

    current_apl = weighted_apl(current)
    previous_apl = weighted_apl(previous)

    delta = None
    if current_apl is not None and previous_apl is not None and previous_apl > 0:
        change_pct = 100 * (current_apl / previous_apl - 1)
        rounded_change = round(change_pct, 2)
        delta = 0 if rounded_change == 0 else f"{rounded_change:+.2f}%"

    population = current["total_population"].sum()
    population_display = (
        f"{population / 1_000_000:.2f}M"
        if population >= 1_000_000
        else f"{population:,.0f}"
    )

    department_count = len(view["departments"])
    declining_count = int(view["trends"]["absolute_change"].lt(0).sum())

    st.subheader("Access in your selected territories")
    st.caption(f"{view['area']} · 2024 indicators, compared with 2023")

    cards = st.columns(4)

    cards[0].metric(
        "2024 potential access",
        f"{current_apl:.2f}" if current_apl is not None else None,
        delta=delta,
        delta_description="vs 2023",
        help="Annual accessible consultations per age-standardised inhabitant.",
        border=True,
    )
    cards[1].metric(
        "Residents (2022)",
        population_display,
        help=f"{population:,.0f} residents. M means million.",
        border=True,
    )
    cards[2].metric(
        "Departments",
        str(department_count),
        border=True,
    )
    cards[3].metric(
        "Declining departments",
        f"{declining_count} / {department_count}",
        help="Selected departments with any negative APL change from 2023 to 2024.",
        border=True,
    )
'''

module_path = PROJECT_DIR / "metrics.py"
compile(metrics_code, str(module_path), "exec")
module_path.write_text(metrics_code, encoding="utf-8")

# Connect this section without duplicating it on reruns.
app_text = APP_PATH.read_text(encoding="utf-8")
import_line = "from metrics import render_metrics"
call_line = "render_metrics(view)"

if import_line not in app_text:
    app_text = app_text.replace(
        "import streamlit as st",
        f"import streamlit as st\n{import_line}",
        1,
    )

if call_line not in app_text:
    app_text += f"\n\n{call_line}\n"

compile(app_text, str(APP_PATH), "exec")
APP_PATH.write_text(app_text, encoding="utf-8")

# Compare the app's figures with our notebook results.
test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."

national_values = [metric.value for metric in test_app.metric]
assert national_values == ["3.26", "67.76M", "100", "79 / 100"]
print("National KPIs:", national_values)
print("National change:", test_app.metric[0].delta)

test_app.sidebar.selectbox[0].set_value("Guyane").run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."

guyane_values = [metric.value for metric in test_app.metric]
assert guyane_values == ["1.41", "288,382", "1", "1 / 1"]
print("Guyane KPIs:", guyane_values)
print("Guyane change:", test_app.metric[0].delta)

2026-10-24 09:48:26.258 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


National KPIs: ['3.26', '67.76M', '100', '79 / 100']
National change: -1.09%
Guyane KPIs: ['1.41', '288,382', '1', '1 / 1']
Guyane change: -8.87%


In [30]:
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlsplit
from google.colab import output
import requests

PREVIEW_PORT = 8501
PROJECT_DIR = Path("/content/apl_dashboard")
APP_PATH = PROJECT_DIR / "app.py"

# Get the browser address provided by Colab.
proxy_url = output.eval_js(
    f"google.colab.kernel.proxyPort({PREVIEW_PORT})"
)
proxy_host = urlsplit(proxy_url).hostname

# Stop our previous preview if this cell is rerun.
if (
    "apl_preview_process" in globals()
    and apl_preview_process.poll() is None
):
    apl_preview_process.terminate()
    apl_preview_process.wait(timeout=10)

preview_log = PROJECT_DIR / "streamlit_preview.log"

with preview_log.open("w", encoding="utf-8") as log_file:
    apl_preview_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", str(APP_PATH),
            f"--server.port={PREVIEW_PORT}",
            "--server.address=0.0.0.0",
            "--server.headless=true",
            "--server.runOnSave=true",
            f"--browser.serverAddress={proxy_host}",
            "--browser.gatherUsageStats=false",
        ],
        cwd=PROJECT_DIR,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

# Wait until the server responds.
for attempt in range(30):
    if apl_preview_process.poll() is not None:
        raise RuntimeError(preview_log.read_text(encoding="utf-8"))

    try:
        health = requests.get(
            f"http://127.0.0.1:{PREVIEW_PORT}/_stcore/health",
            timeout=1,
        )
        if health.status_code == 200:
            break
    except requests.RequestException:
        pass

    time.sleep(0.5)
else:
    raise RuntimeError("Startup timed out. Check streamlit_preview.log.")

output.serve_kernel_port_as_iframe(PREVIEW_PORT, height=900)

<IPython.core.display.Javascript object>

In [31]:
# Reuse this helper to connect each new section to app.py.
def connect_section(module_name, function_name):
    source = APP_PATH.read_text(encoding="utf-8")
    import_line = f"from {module_name} import {function_name}"
    call_line = f"{function_name}(view)"

    if import_line not in source:
        source = source.replace(
            "import streamlit as st",
            f"import streamlit as st\n{import_line}",
            1,
        )
    if call_line not in source:
        source += f"\n\n{call_line}\n"

    compile(source, str(APP_PATH), "exec")
    APP_PATH.write_text(source, encoding="utf-8")


ranking_code = '''import plotly.express as px
import streamlit as st


def render_ranking(view):
    departments = view["departments"]
    metric = "apl_65_or_younger"

    national_apl = float(
        view["national"].loc[
            view["national"]["apl_year"].eq(2024), "weighted_apl"
        ].iloc[0]
    )

    st.subheader("1. Compare departments")
    order = st.radio(
        "Show departments with",
        ["Lowest APL", "Highest APL"],
        horizontal=True,
        key="ranking_order",
    )

    ranked = (
        departments.nsmallest(10, metric)
        if order == "Lowest APL"
        else departments.nlargest(10, metric)
    )

    fig = px.bar(
        ranked,
        x=metric,
        y="department_name",
        orientation="h",
        text_auto=".2f",
        hover_data={
            metric: ":.3f",
            "department_code": True,
            "total_population": ":,.0f",
        },
        labels={
            metric: "APL (consultations/year/standardised inhabitant)",
            "department_name": "Department",
            "department_code": "Department code",
            "total_population": "Population (2022)",
        },
        title=(
            f"{order} among selected departments, 2024"
            "<br><sup>GPs aged 65 or younger. Source: DREES.</sup>"
        ),
        color_discrete_sequence=["#2E86C1"],
        range_x=[0, 1.2 * max(national_apl, ranked[metric].max())],
        height=max(340, 40 * len(ranked) + 140),
    )

    fig.update_yaxes(autorange="reversed", automargin=True)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.add_vline(
        x=national_apl,
        line_dash="dash",
        line_color="#808080",
        annotation_text=f"National APL: {national_apl:.2f}",
        annotation_position="top left",
    )

    st.plotly_chart(
        fig,
        width="stretch",
        theme="streamlit",
        key="department_ranking",
        config={"displaylogo": False, "scrollZoom": False},
        alt="Horizontal bars comparing departmental APL with the national benchmark.",
    )

    focus = ranked.iloc[0]
    st.write(
        f"**{focus['department_name']}**: APL **{focus[metric]:.2f}**, "
        f"compared with **{national_apl:.2f}** nationally."
    )
    st.caption(
        f"{len(ranked)} of {len(departments)} selected departments shown."
    )
'''

ranking_path = PROJECT_DIR / "ranking.py"
compile(ranking_code, str(ranking_path), "exec")
ranking_path.write_text(ranking_code, encoding="utf-8")
connect_section("ranking", "render_ranking")

test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."
print("App execution check passed.")
print("Plotly charts:", len(test_app.get("plotly_chart")))

2026-10-24 09:48:37.545 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App execution check passed.
Plotly charts: 1


In [32]:
import shutil

# Include the previously downloaded boundaries in the app folder.
shutil.copy2(
    GEOJSON_PATH,
    PROJECT_DIR / "assets" / "departments_2024.geojson",
)

map_code = '''import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st


@st.cache_data(show_spinner=False)
def load_map_assets(base_dir):
    boundaries = json.loads(
        (base_dir / "assets" / "departments_2024.geojson")
        .read_text(encoding="utf-8")
    )
    national_data = pd.read_csv(
        base_dir / "data" / "processed" / "apl_departments_2024.csv",
        usecols=["apl_65_or_younger"],
    )
    values = national_data["apl_65_or_younger"]
    return boundaries, [float(values.min()), float(values.max())]


def render_map(view):
    base_dir = Path(__file__).resolve().parent
    boundaries, colour_range = load_map_assets(base_dir)
    departments = view["departments"]
    selected_codes = set(departments["department_code"])

    area_codes = {
        "Metropolitan France": {
            code for code in selected_codes if not code.startswith("97")
        },
        "Guadeloupe": {"971"},
        "Martinique": {"972"},
        "Guyane": {"973"},
        "La Réunion": {"974"},
    }
    available = [
        area for area, codes in area_codes.items()
        if codes & selected_codes
    ]

    st.subheader("2. Explore geographic differences")

    focus = st.selectbox(
        "Map focus",
        available,
        key="map_focus_" + "|".join(available),
        disabled=len(available) == 1,
    )

    map_data = departments.loc[
        departments["department_code"].isin(area_codes[focus])
    ]
    map_codes = set(map_data["department_code"])
    map_geojson = {
        "type": "FeatureCollection",
        "features": [
            feature for feature in boundaries["features"]
            if feature["properties"]["code"] in map_codes
        ],
    }

    fig = px.choropleth(
        map_data,
        geojson=map_geojson,
        locations="department_code",
        featureidkey="properties.code",
        color="apl_65_or_younger",
        hover_name="department_name",
        hover_data={
            "apl_65_or_younger": ":.3f",
            "department_code": True,
            "total_population": ":,.0f",
        },
        labels={
            "apl_65_or_younger": "APL",
            "department_code": "Department code",
            "total_population": "Population (2022)",
        },
        color_continuous_scale="Cividis",
        range_color=colour_range,
        projection="mercator",
        title=(
            f"Potential GP access in {focus}, 2024"
            "<br><sup>GPs aged 65 or younger. Source: DREES.</sup>"
        ),
        height=550,
    )

    fig.update_geos(
        fitbounds="locations",
        visible=False,
        bgcolor="rgba(0,0,0,0)",
    )
    fig.update_traces(marker_line_color="white", marker_line_width=0.5)
    fig.update_layout(
        coloraxis_colorbar={
            "title": "Consultations/year<br>per standardised<br>inhabitant"
        }
    )

    st.plotly_chart(
        fig,
        width="stretch",
        theme="streamlit",
        key="department_map",
        config={"displaylogo": False, "scrollZoom": False},
        alt=f"Map of departmental potential GP access in {focus}, 2024.",
    )

    st.caption(
        f"{len(map_data)} of {len(departments)} selected departments shown. "
        "The national colour scale stays fixed across views."
    )
    st.caption(
        "[Boundaries: Etalab/IGN]"
        "(https://www.data.gouv.fr/datasets/contours-administratifs) · ODbL."
    )
'''

map_path = PROJECT_DIR / "map_view.py"
compile(map_code, str(map_path), "exec")
map_path.write_text(map_code, encoding="utf-8")
connect_section("map_view", "render_map")

test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."
print("App execution check passed.")
print("Plotly charts:", len(test_app.get("plotly_chart")))

2026-10-24 09:48:39.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App execution check passed.
Plotly charts: 2


In [33]:
local_code = '''import plotly.express as px
import streamlit as st

from metrics import weighted_apl


def render_local(view):
    st.subheader("3. Look beyond the department average")

    departments = view["departments"].sort_values("apl_65_or_younger")
    names = departments.set_index("department_code")["department_name"].to_dict()
    codes = departments["department_code"].tolist()

    selected_code = st.selectbox(
        "Department to explore",
        options=codes,
        format_func=lambda code: f"{names[code]} ({code})",
        key="local_department_" + "|".join(codes),
    )
    department_name = names[selected_code]

    communes = view["communes"]
    local = communes.loc[
        communes["apl_year"].eq(2024)
        & communes["department_code"].eq(selected_code)
    ].copy()

    department_apl = weighted_apl(local)
    national_apl = float(
        view["national"].loc[
            view["national"]["apl_year"].eq(2024), "weighted_apl"
        ].iloc[0]
    )

    population = local["total_population"].sum()
    zero_apl = local["apl_65_or_younger"].eq(0)
    zero_population = local.loc[zero_apl, "total_population"].sum()

    below_national = local["apl_65_or_younger"].lt(national_apl)
    below_population = local.loc[
        below_national, "total_population"
    ].sum()

    fig = px.histogram(
        local,
        x="apl_65_or_younger",
        y="total_population",
        histfunc="sum",
        histnorm="percent",
        nbins=15,
        color_discrete_sequence=["#2E86C1"],
        title=(
            f"Where residents fall on the access spectrum: {department_name}"
            "<br><sup>2024 APL · Population reference: 2022 · "
            "GPs aged 65 or younger</sup>"
        ),
        height=430,
    )

    fig.update_traces(
        marker_line_width=0.5,
        marker_line_color="white",
        hovertemplate=(
            "APL bin: %{x}<br>"
            "Share of department residents: %{y:.1f}%"
            "<extra></extra>"
        ),
    )
    fig.update_layout(
        bargap=0.05,
        xaxis_title=(
            "Annual accessible consultations per age-standardised inhabitant"
        ),
        yaxis_title="Share of department residents (%)",
    )
    fig.update_xaxes(rangemode="tozero")
    fig.update_yaxes(rangemode="tozero")

    fig.add_vline(
        x=department_apl,
        line_dash="dash",
        line_color="#808080",
        annotation_text=f"Department average: {department_apl:.2f}",
        annotation_position="top right",
    )

    st.plotly_chart(
        fig,
        width="stretch",
        theme="streamlit",
        key="local_distribution",
        config={"displaylogo": False, "scrollZoom": False},
        alt=(
            f"Population-weighted distribution of commune-level APL "
            f"in {department_name}, with the department average marked."
        ),
    )

    st.caption(
        "Each bar groups geographical units with similar APL. "
        "Its height represents their share of department residents."
    )

    st.write(
        f"**{int(zero_apl.sum())} of {len(local)} geographical units** "
        f"have zero APL for GPs aged 65 or younger, covering "
        f"**{zero_population:,.0f} residents "
        f"({100 * zero_population / population:.1f}%)**."
    )
    st.write(
        f"**{100 * below_population / population:.1f}% of residents** "
        f"live in geographical units below the national APL benchmark "
        f"of **{national_apl:.2f}**."
    )
    st.caption(
        "Zero APL does not establish an absence of all healthcare. "
        "The national average is a comparison benchmark, "
        "not an official shortage threshold."
    )
'''

local_path = PROJECT_DIR / "local_view.py"
compile(local_code, str(local_path), "exec")
local_path.write_text(local_code, encoding="utf-8")

connect_section("local_view", "render_local")

test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."

print("App execution check passed.")
print("Plotly charts:", len(test_app.get("plotly_chart")))

2026-10-24 09:48:42.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App execution check passed.
Plotly charts: 3


In [34]:
trend_code = '''import plotly.express as px
import streamlit as st


def render_trends(view):
    st.subheader("4. Is access improving?")

    trends = view["trends"].copy()
    national_apl = float(
        view["national"].loc[
            view["national"]["apl_year"].eq(2024), "weighted_apl"
        ].iloc[0]
    )

    trends["direction"] = trends["absolute_change"].map(
        lambda value: (
            "Increase" if value > 0
            else "Decrease" if value < 0
            else "No change"
        )
    )

    fig = px.scatter(
        trends,
        x="apl_2024",
        y="relative_change_pct",
        color="direction",
        symbol="direction",
        color_discrete_map={
            "Increase": "#2E86C1",
            "Decrease": "#E69F00",
            "No change": "#808080",
        },
        symbol_map={
            "Increase": "circle",
            "Decrease": "triangle-down",
            "No change": "square",
        },
        hover_name="department_name",
        hover_data={
            "department_code": True,
            "apl_2024": ":.3f",
            "apl_2023": ":.3f",
            "relative_change_pct": ":+.2f",
            "absolute_change": ":+.3f",
            "direction": False,
        },
        labels={
            "apl_2024": "2024 potential access (APL)",
            "apl_2023": "2023 potential access (APL)",
            "relative_change_pct": "Change from 2023 to 2024 (%)",
            "absolute_change": "Absolute APL change",
            "department_code": "Department code",
            "direction": "Change direction",
        },
        title=(
            "Access levels and recent changes"
            "<br><sup>One point per selected department · "
            "GPs aged 65 or younger · Source: DREES</sup>"
        ),
        height=500,
    )

    fig.update_traces(
        marker={
            "size": 11,
            "opacity": 0.8,
            "line": {"width": 0.6, "color": "white"},
        }
    )

    lower = min(float(trends["relative_change_pct"].min()), 0)
    upper = max(float(trends["relative_change_pct"].max()), 0)
    padding = max(upper - lower, 1) * 0.12

    fig.update_yaxes(range=[lower - padding, upper + padding])
    fig.update_xaxes(
        range=[0, max(national_apl, trends["apl_2024"].max()) * 1.15]
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="#808080",
    )
    fig.add_vline(
        x=national_apl,
        line_dash="dash",
        line_color="#808080",
        annotation_text=f"National APL: {national_apl:.2f}",
        annotation_position="top right",
    )

    st.plotly_chart(
        fig,
        width="stretch",
        theme="streamlit",
        key="department_trends",
        config={"displaylogo": False, "scrollZoom": False},
        alt=(
            "Scatter plot comparing departmental APL in 2024 with "
            "its percentage change since 2023. Reference lines mark "
            "the national APL and zero change."
        ),
    )

    st.caption(
        "Lower-left: below the national APL benchmark and declining. "
        "Colours and symbols distinguish increases from decreases."
    )

    declining = int(trends["absolute_change"].lt(0).sum())
    increasing = int(trends["absolute_change"].gt(0).sum())

    st.write(
        f"**{declining} of {len(trends)} selected departments** "
        f"recorded a decrease; **{increasing}** recorded an increase."
    )

    both_conditions = trends.loc[
        trends["apl_2024"].lt(national_apl)
        & trends["absolute_change"].lt(0)
    ]

    departments = view["departments"]
    residents = departments.loc[
        departments["department_code"].isin(
            both_conditions["department_code"]
        ),
        "total_population",
    ].sum()
    selected_population = departments["total_population"].sum()
    share = 100 * residents / selected_population

    st.write(
        f"**{len(both_conditions)} departments** combine below-national "
        f"APL in 2024 with a decline since 2023. "
        f"**{residents:,.0f} residents** live in these departments "
        f"(**{share:.1f}%** of the selected population, reference year 2022)."
    )

    st.caption(
        "Department averages can hide local differences. "
        "These resident counts do not mean every individual has low access. "
        "Comparisons use each year's full dataset and population weights; "
        "they do not establish the causes of change."
    )
'''

trend_path = PROJECT_DIR / "trend_view.py"
compile(trend_code, str(trend_path), "exec")
trend_path.write_text(trend_code, encoding="utf-8")

connect_section("trend_view", "render_trends")

test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."

print("App execution check passed.")
print("Plotly charts:", len(test_app.get("plotly_chart")))

2026-10-24 09:48:46.258 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App execution check passed.
Plotly charts: 4


In [35]:
import json

apl_source = (
    "https://www.data.gouv.fr/datasets/"
    "accessibilite-potentielle-localisee-apl-aux-professionnels-de-sante"
)
apl_release = (
    "https://drees.solidarites-sante.gouv.fr/"
    "communique-de-presse-jeux-de-donnees/jeux-de-donnees/"
    "260722-accessibilite-aux-soins-de-premier-recours-en-2024"
)
tele_source = (
    "https://drees.solidarites-sante.gouv.fr/"
    "publications-communique-de-presse/etudes-et-resultats/"
    "260212-recours-aux-t%C3%A9l%C3%A9consultations"
)
policy_source = (
    "https://sante.gouv.fr/grands-dossiers/"
    "un-meilleur-acces-aux-soins-pour-tous-sur-le-territoire/article/"
    "pacte-de-lutte-contre-les-deserts-medicaux-151-zones-prioritaires-identifiees"
)
france_sante_source = (
    "https://bulletins-officiels.social.gouv.fr/"
    "circulaire-interministerielle-ndeg-dgossdasas2dgclsdcatc3202686-du-16-juin-2026-"
    "relative-au-deploiement-et-au-financement-du-reseau-france-sante"
)

context_sections = {
    "How to interpret APL": """
- Scope: France excluding Mayotte; GPs aged 65 or younger.
- APL measures potential access to consultations, home visits and
  teleconsultations, accounting for nearby supply, activity and population needs.
- Department and national averages use standardised population as weights:
  **weighted APL = sum(APL × standardised population) /
  sum(standardised population)**.
- Resident counts and population shares use actual population.
- Population references are 2022 for APL 2024 and 2021 for APL 2023.
- The map focus changes the displayed geography; sidebar filters determine
  the analytical selection.
""",
    "Data quality and limitations": """
- The prepared tables contain 34,890 geographical units in 2024 and
  34,954 in 2023, with no missing values or duplicate commune codes within a year.
- Zero APL values are retained. They do not establish an absence of all healthcare.
- Three zero-population observations in 2023 are retained and flagged;
  they contribute no weight to population-weighted averages.
- High APL values are retained. Excluding the five highest 2024 observations
  changes national APL from 3.258882 to 3.258369.
- Commune codes differ between years. Comparisons use each year's full dataset,
  rather than treating unmatched codes as lost or newly opened practices.
- APL does not directly measure appointment waiting times, acceptance of new
  patients, affordability or quality of care.
- Department averages can hide differences between communes.
  Below-national APL is not an official medical-desert classification.
- Changes are descriptive. They do not establish causes or predict retirements.
  Population standardisation does not fully account for changing healthcare
  needs caused by national population ageing between years.
""",
    "Teleconsultations and digital access": f"""
**Teleconsultations are included in the activity underlying APL.**
This workbook does not separate remote from in-person care or identify
platforms such as Doctolib. It cannot show how much remote care compensates
for geographical isolation or limited appointment availability.

A separate DREES study of adults in metropolitan France reports that, in 2024,
21% of residents in large urban centres used teleconsultation, compared with
around 10% in rural areas. Among teleconsultation users, 43% cited long waits
for an in-person appointment. The study covers GPs and specialists.

These findings suggest that remote care is relevant to access, but do not
support assuming that the most isolated residents use it most.

[Source: DREES, Études et Résultats 1366, February 2026]({tele_source}).
""",
    "Policy context — checked 23 October 2026": f"""
The Ministry describes **Un médecin près de chez vous**, operating since
September 2025 in 153 priority territories, with voluntary participation
by doctors in its pilot implementation.
[Ministry programme description]({policy_source}).

The **France Santé** circular of 16 June 2026 sets out the network's deployment
and funding, with a target of 5,000 labelled structures by 2027.
This is a target, not a result established by this dashboard.
[Official circular]({france_sante_source}).

**The 2023–2024 observations predate these initiatives and cannot evaluate
their effectiveness.** Later data and a dedicated evaluation would be needed.
""",
    "Sources and licences": f"""
- [Main APL dataset: DREES, referenced on data.gouv.fr]({apl_source})
  — Open Licence 2.0.
- [DREES publication on access in 2024]({apl_release}).
- [Department boundaries: Etalab/IGN, 2024 edition]
  (https://www.data.gouv.fr/datasets/contours-administratifs)
  — ODbL.
- [Department names: French geographic API]
  (https://geo.api.gouv.fr/departements).

Teleconsultation and policy references provide context; they are not
additional observations in the APL analysis.
""",
}

# Keep Markdown links on one line.
context_sections = {
    title: text.replace("]\n  (", "](")
    for title, text in context_sections.items()
}

context_path = PROJECT_DIR / "assets" / "context_notes.json"
context_path.write_text(
    json.dumps(context_sections, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

notes_path = PROJECT_DIR / "CONTEXT_AND_LIMITATIONS.md"
notes_path.write_text(
    "\n\n".join(
        f"## {title}\n\n{text.strip()}"
        for title, text in context_sections.items()
    ),
    encoding="utf-8",
)

context_code = '''import json
from pathlib import Path

import streamlit as st


def render_context(view):
    st.subheader("5. From evidence to local action")
    st.caption(
        "For municipalities, healthcare centres and regional health authorities."
    )

    st.write(
        "Start with departments combining below-national access and decline, "
        "then inspect the differences between their communes. "
        "Use these signals to guide further local investigation."
    )
    st.markdown(
        "- Check waiting times and whether practices accept new patients.\\n"
        "- Discuss staffing, transport and digital access with local services.\\n"
        "- Follow later APL releases before assessing policy outcomes."
    )

    st.caption(
        f"Current selection: {len(view['departments'])} departments. "
        "The methodological notes below apply to every selection."
    )

    path = Path(__file__).resolve().parent / "assets" / "context_notes.json"
    sections = json.loads(path.read_text(encoding="utf-8"))

    for title, text in sections.items():
        with st.expander(title):
            st.markdown(text)
'''

module_path = PROJECT_DIR / "context_view.py"
compile(context_code, str(module_path), "exec")
module_path.write_text(context_code, encoding="utf-8")

connect_section("context_view", "render_context")

test_app = AppTest.from_file(str(APP_PATH)).run(timeout=30)
assert not test_app.exception, "Inspect the app error shown above."

print("App execution check passed.")
print("Plotly charts:", len(test_app.get("plotly_chart")))
print("Context notes saved for the app and report.")

2026-10-24 09:48:48.483 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App execution check passed.
Plotly charts: 4
Context notes saved for the app and report.


In [36]:
from streamlit.testing.v1 import AppTest

app_check = AppTest.from_file(str(APP_PATH)).run(timeout=30)
check_results = []


def check_selection(label, expected_metrics, expected_map, expected_delta=None):
    assert not app_check.exception, f"{label}: app execution failed."

    values = [metric.value for metric in app_check.metric[:4]]

    for index, expected in expected_metrics.items():
        assert values[index] == expected, (
            f"{label}: metric {index} returned {values[index]!r}, "
            f"expected {expected!r}."
        )

    if expected_delta is not None:
        actual_delta = app_check.metric[0].delta
        assert actual_delta == expected_delta, (
            f"{label}: change returned {actual_delta!r}, "
            f"expected {expected_delta!r}."
        )

    charts = len(app_check.get("plotly_chart"))
    assert charts == 4, f"{label}: expected 4 charts, found {charts}."

    local_selector = next(
        widget for widget in app_check.selectbox
        if widget.label == "Department to explore"
    )
    assert len(local_selector.options) == int(values[2]), (
        f"{label}: local department choices do not match the selection."
    )

    map_selector = next(
        widget for widget in app_check.selectbox
        if widget.label == "Map focus"
    )
    assert map_selector.value == expected_map, (
        f"{label}: unexpected map focus."
    )

    check_results.append({
        "Selection": label,
        "APL": values[0],
        "Residents": values[1],
        "Departments": values[2],
        "Declining": values[3],
        "Charts": charts,
        "Status": "Passed",
    })


# Check the national default against the validated analysis.
check_selection(
    "France excluding Mayotte",
    {0: "3.26", 1: "67.76M", 2: "100", 3: "79 / 100"},
    expected_map="Metropolitan France",
    expected_delta="-1.09%",
)

# Check an overseas selection.
app_check.sidebar.selectbox[0].set_value("Guyane").run(timeout=30)
check_selection(
    "Guyane",
    {0: "1.41", 1: "288,382", 2: "1", 3: "1 / 1"},
    expected_map="Guyane",
    expected_delta="-8.87%",
)

# Check that returning to metropolitan France restores its department options.
app_check.sidebar.selectbox[0].set_value(
    "Metropolitan France"
).run(timeout=30)
check_selection(
    "Metropolitan France",
    {2: "96"},
    expected_map="Metropolitan France",
)

# Check the department multiselect independently of the area filter.
app_check.sidebar.multiselect[0].set_value(["77"]).run(timeout=30)
check_selection(
    "Seine-et-Marne only",
    {0: "1.94", 1: "1.45M", 2: "1", 3: "1 / 1"},
    expected_map="Metropolitan France",
    expected_delta="-4.06%",
)

display(pd.DataFrame(check_results))
print("All four filter scenarios passed.")

2026-10-24 09:48:50.527 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,Selection,APL,Residents,Departments,Declining,Charts,Status
0,France excluding Mayotte,3.26,67.76M,100,79 / 100,4,Passed
1,Guyane,1.41,"288,382",1,1 / 1,4,Passed
2,Metropolitan France,3.25,65.85M,96,77 / 96,4,Passed
3,Seine-et-Marne only,1.94,1.45M,1,1 / 1,4,Passed


All four filter scenarios passed.


In [37]:
import hashlib
import json
import shutil
import subprocess
import sys

# Include the original workbook in the project.
project_raw = PROJECT_DIR / "data" / "raw"
project_raw.mkdir(parents=True, exist_ok=True)

workbook_copy = project_raw / "apl_general_practitioners.xlsx"
shutil.copy2(FILE_PATH, workbook_copy)

# Record where each source came from.
resources = [
    {
        "name": "DREES APL workbook",
        "path": "data/raw/apl_general_practitioners.xlsx",
        "url": (
            "https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/"
            "datasets/530_l-accessibilite-potentielle-localisee-apl/"
            "attachments/indicateur_d_apl_aux_medecins_generalistes_xlsx"
        ),
        "licence": "Open Licence 2.0",
    },
    {
        "name": "Department boundaries, 2024",
        "path": "assets/departments_2024.geojson",
        "url": (
            "https://etalab-datasets.geo.data.gouv.fr/"
            "2024/geojson/departements-1000m.geojson"
        ),
        "licence": "ODbL",
    },
    {
        "name": "EFREI logo",
        "path": "assets/efrei_logo.png",
        "url": (
            "https://www.efrei.fr/wp-content/uploads/"
            "2025/09/Logo-Efrei-CMJN-scaled.png"
        ),
        "licence": "EFREI branding; not covered by the data licences",
    },
]

for resource in resources:
    path = PROJECT_DIR / resource["path"]
    assert path.is_file(), f"Missing source file: {path}"

    with path.open("rb") as file:
        resource["sha256"] = hashlib.file_digest(
            file, "sha256"
        ).hexdigest()

manifest = {
    "snapshot_recorded_on": "2026-10-23",
    "apl_years_used": [2023, 2024],
    "resources": resources,
}

(PROJECT_DIR / "data_sources.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

download_code = '''import hashlib
import json
from pathlib import Path

import requests


BASE_DIR = Path(__file__).resolve().parent


def file_hash(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


def fetch_resource(resource):
    target = BASE_DIR / resource["path"]
    expected_hash = resource["sha256"]

    if target.is_file():
        if file_hash(target) != expected_hash:
            raise ValueError(
                f"Existing file differs from the recorded snapshot: {target}"
            )
        print(f"Using verified cached file: {resource['path']}")
        return

    target.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(resource["url"], timeout=60)
    response.raise_for_status()

    temporary = target.with_suffix(target.suffix + ".part")
    temporary.write_bytes(response.content)

    if file_hash(temporary) != expected_hash:
        temporary.unlink()
        raise ValueError(
            f"The online resource differs from the recorded snapshot: "
            f"{resource['name']}. Use the submitted snapshot or investigate "
            f"the source update before rebuilding the analysis."
        )

    temporary.replace(target)
    print(f"Downloaded and verified: {resource['path']}")


def main():
    manifest = json.loads(
        (BASE_DIR / "data_sources.json").read_text(encoding="utf-8")
    )
    for resource in manifest["resources"]:
        fetch_resource(resource)


if __name__ == "__main__":
    main()
'''

download_path = PROJECT_DIR / "download_data.py"
compile(download_code, str(download_path), "exec")
download_path.write_text(download_code, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(download_path)],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)

assert result.returncode == 0, result.stderr
print(result.stdout.strip())
print("Source manifest saved: data_sources.json")
print("Cached downloader check passed.")

Using verified cached file: data/raw/apl_general_practitioners.xlsx
Using verified cached file: assets/departments_2024.geojson
Using verified cached file: assets/efrei_logo.png
Source manifest saved: data_sources.json
Cached downloader check passed.


In [38]:
import json
import platform

metadata = {
    "author": "Yanis AMARA",
    "student_number": "20232166",
    "group": "DE2",
    "email": "Yanis.amara@efrei.net",
    "institution": "EFREI",
    "programme": "DATA ENGINEERING",
    "academic_year": "2026-2027",
    "course": "Data Analysis & Visualization",
    "professor": "Dr. Mano MATHEW",
    "linkedin": "https://www.linkedin.com/in/yanis-amara-3a3953292/",
    "python_version": platform.python_version(),
    "archive_name": "StApp26_20232166_AMARA_DE2.zip",
    "app_url": None,
    "repository_url": None,
    "demo_video_url": None,
}

(PROJECT_DIR / "project_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

requirements = (PROJECT_DIR / "requirements.txt").read_text(
    encoding="utf-8"
).strip()

readme = rf"""# Access to General Practitioners in France

A Streamlit dashboard exploring geographical differences in potential access
to general practitioners in 2024 and changes since 2023.

This project is motivated by my upcoming healthcare research internship
and an interest in territorial inequalities in access to care.

## Author

- **Name:** {metadata["author"]}
- **Student number / group:** {metadata["student_number"]} / {metadata["group"]}
- **Institution:** {metadata["institution"]}
- **Programme:** {metadata["programme"]}
- **Academic year:** {metadata["academic_year"]}
- **Course:** {metadata["course"]}
- **Professor:** {metadata["professor"]}
- **Email:** {metadata["email"]}
- **LinkedIn:** {metadata["linkedin"]}

## Audience and questions

The dashboard is intended for municipalities, healthcare centres and
regional health authorities.

**Main question:** How does potential access to GPs vary across France?

1. Which departments have the lowest or highest potential access in 2024?
2. What differences are hidden within department averages?
3. Where did potential access decline between 2023 and 2024?

## Main findings

These figures refer to the complete national selection, excluding Mayotte.

- Population-weighted APL decreased from **3.295 in 2023 to 3.259 in 2024**,
  a change of **-1.09%**.
- **79 of 100 departments** recorded a decrease.
- **50 departments** combined below-national APL in 2024 with a decline.
  They contained **25,788,500 residents**, or **38.1%** of the covered
  population, using the 2022 population reference.
- In Guyane, **12 of 22 communes** had zero APL for GPs aged 65 or younger.
  These communes contained **47,077 residents**, or **16.3%** of the
  department population.

Living in a qualifying department does not mean that every resident
experiences low access. Zero APL does not establish an absence of all healthcare.

## Dashboard walkthrough

1. **Overview:** KPIs respond to the sidebar area and department filters.
2. **Compare departments:** switch between the lowest and highest APL rankings.
3. **Explore geography:** inspect the map, including overseas departments.
4. **Look within a department:** explore the population-weighted distribution
   of commune-level access.
5. **Examine change:** compare 2024 access with its percentage change since 2023.
6. **Interpret the evidence:** read recommendations, methods and limitations.

The map-focus control changes the displayed geography only.
Sidebar filters determine the analytical selection.
The map colour scale remains fixed across views.

## Data and methods

**Main source:** DREES Localised Potential Accessibility (APL), referenced on
[data.gouv.fr]({apl_source}).

Scope: France excluding Mayotte; GPs aged 65 or younger.
APL describes annual accessible consultations, home visits and teleconsultations
per age-standardised inhabitant.

- National and department APL averages use **standardised population weights**.
- Resident counts and population shares use **actual population**.
- APL 2024 uses the **2022 population reference**; APL 2023 uses **2021**.
- Each year is aggregated using its complete set of observations.
- The national average is a comparison benchmark, not an official
  medical-desert threshold.

Detailed preparation and interpretation:
[DATA_PREPARATION.md](DATA_PREPARATION.md) and
[CONTEXT_AND_LIMITATIONS.md](CONTEXT_AND_LIMITATIONS.md).

Source URLs, licences and SHA-256 fingerprints are recorded in
[data_sources.json](data_sources.json).

## Run locally

Run the following commands from the project directory.
The Colab development environment used **Python {metadata["python_version"]}**.

### Windows PowerShell

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe download_data.py
.\.venv\Scripts\python.exe -m streamlit run app.py
```

### macOS / Linux

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python download_data.py
.venv/bin/python -m streamlit run app.py
```

The application reads the included processed CSV snapshots.
The downloader reuses verified source files and retrieves missing files.
If an online resource has changed since the recorded snapshot, verification
stops rather than silently substituting new data.

### Pinned dependencies

```text
{requirements}
```

## Project organisation

- `app.py`: application entry point.
- `data_and_filters.py`: cached loading and sidebar filters.
- `metrics.py`: weighted calculations and KPI cards.
- `ranking.py`, `map_view.py`, `local_view.py`, `trend_view.py`: interactive charts.
- `context_view.py`: recommendations and explanatory notes.
- `data/raw/`: original workbook.
- `data/processed/`: prepared tables and numerical audit.
- `assets/`: department boundaries, EFREI logo and context notes.
- `download_data.py`: cached source downloader with fingerprint verification.
- `requirements.txt`: pinned application dependencies.
- `project_metadata.json`: author details and submission links.

## Validation status

Streamlit AppTest passed four filter scenarios:

- France excluding Mayotte.
- Guyane.
- Metropolitan France.
- Seine-et-Marne selected through the department filter.

Checks cover expected KPIs, percentage changes, chart count,
department options and map focus.

A full visual review and a clean-environment check remain to be completed.
AppTest execution checks alone do not establish visual correctness.

## Limits and context

APL measures potential access, not actual waiting times, acceptance of new
patients or quality of care. Department averages can conceal local disparities.

Teleconsultations are included in APL activity, but the workbook does not
separate their contribution or measure the effect of platforms such as Doctolib.

The 2023-2024 observations predate the later policy initiatives discussed
in the context notes and cannot establish their effectiveness.

## Licences and attribution

- DREES APL data: **Open Licence 2.0**.
- Department boundaries, Etalab/IGN: **ODbL**.
- EFREI logo: institutional branding, separate from the data licences.

The project uses aggregate territorial data and contains no patient-level data.

## Submission checklist

- [x] Streamlit application with four interactive charts and sidebar filters.
- [x] Source files, download script and source manifest.
- [x] Pinned requirements and local run instructions.
- [ ] Export the completed Colab notebook as `analysis.ipynb`.
- [ ] Complete the clean-environment check and visual review.
- [ ] Add the final report and storyboard.
- [ ] Add the repository URL.
- [ ] Add the validated deployment URL.
- [ ] Record the 2-4 minute demo and add its link.
- [ ] Create `{metadata["archive_name"]}`.

**App URL:** TO_BE_ADDED
**Repository URL:** TO_BE_ADDED
**Demo video URL:** TO_BE_ADDED
"""

(PROJECT_DIR / "README.md").write_text(
    readme.strip() + "\n",
    encoding="utf-8",
)

gitignore = """__pycache__/
*.py[cod]
.venv/
.ipynb_checkpoints/
*.log
.streamlit/secrets.toml
.DS_Store
"""

(PROJECT_DIR / ".gitignore").write_text(
    gitignore,
    encoding="utf-8",
)

print("Saved: project_metadata.json")
print("Saved: README.md")
print("Saved: .gitignore")
print("Final archive:", metadata["archive_name"])

Saved: project_metadata.json
Saved: README.md
Saved: .gitignore
Final archive: StApp26_20232166_AMARA_DE2.zip


In [39]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import venv
from pathlib import Path

# Save a reusable check that also works outside Colab.
tests_dir = PROJECT_DIR / "tests"
tests_dir.mkdir(exist_ok=True)

check_code = '''import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

from streamlit.testing.v1 import AppTest

BASE_DIR = Path(__file__).resolve().parents[1]

installed_versions = {}
for line in (BASE_DIR / "requirements.txt").read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    package, expected = line.split("==")
    actual = version(package)
    assert actual == expected, f"{package}: {actual} != {expected}"
    installed_versions[package] = actual

scenarios = [
    (
        "National", "France excluding Mayotte", None,
        ["3.26", "67.76M", "100", "79 / 100"], "-1.09%",
    ),
    (
        "Guyane", "Guyane", None,
        ["1.41", "288,382", "1", "1 / 1"], "-8.87%",
    ),
    (
        "Metropolitan France", "Metropolitan France", None,
        ["3.25", "65.85M", "96", "77 / 96"], None,
    ),
    (
        "Seine-et-Marne", "Metropolitan France", ["77"],
        ["1.94", "1.45M", "1", "1 / 1"], "-4.06%",
    ),
]

app = AppTest.from_file(str(BASE_DIR / "app.py")).run(timeout=60)
results = []

for label, area, departments, expected, delta in scenarios:
    assert not app.exception, f"{label}: app execution failed."

    if app.sidebar.selectbox[0].value != area:
        app.sidebar.selectbox[0].set_value(area).run(timeout=60)

    if departments is not None:
        app.sidebar.multiselect[0].set_value(departments).run(timeout=60)

    assert not app.exception, f"{label}: filter execution failed."

    values = [metric.value for metric in app.metric[:4]]
    assert values == expected, f"{label}: {values} != {expected}"

    if delta is not None:
        assert app.metric[0].delta == delta, f"{label}: unexpected change."

    assert len(app.get("plotly_chart")) == 4, f"{label}: missing chart."

    local_selector = next(
        widget for widget in app.selectbox
        if widget.label == "Department to explore"
    )
    assert len(local_selector.options) == int(expected[2])

    results.append({"selection": label, "status": "passed"})
    print(f"Passed: {label}")

report = {
    "python_version": platform.python_version(),
    "isolated_environment": sys.prefix != sys.base_prefix,
    "pinned_versions": installed_versions,
    "functional_checks": results,
    "visual_review": "Full dashboard review still pending",
}

(BASE_DIR / "VALIDATION.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)
'''

(tests_dir / "check_app.py").write_text(check_code, encoding="utf-8")


def run_checked(command, cwd, environment):
    result = subprocess.run(
        command,
        cwd=cwd,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=600,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])
    return result.stdout


# Prevent the new environment from inheriting notebook import paths.
child_environment = os.environ.copy()
child_environment.pop("PYTHONPATH", None)
child_environment["PYTHONNOUSERSITE"] = "1"

with tempfile.TemporaryDirectory(prefix="apl_clean_") as temporary:
    temporary = Path(temporary)
    clean_project = temporary / "project"
    clean_environment = temporary / "environment"

    shutil.copytree(
        PROJECT_DIR,
        clean_project,
        ignore=shutil.ignore_patterns(
            "__pycache__", "*.pyc", "*.log", ".venv",
            ".git", ".ipynb_checkpoints",
        ),
    )

    venv.EnvBuilder(
        with_pip=False,
        system_site_packages=False,
    ).create(clean_environment)

    clean_python = clean_environment / (
        "Scripts/python.exe" if os.name == "nt" else "bin/python"
    )
    pip_command = [
        sys.executable, "-m", "pip", "--python", str(clean_python)
    ]

    print("Installing pinned dependencies in an isolated environment...", flush=True)
    run_checked(
        pip_command + [
            "install", "--disable-pip-version-check",
            "-r", str(clean_project / "requirements.txt"),
        ],
        clean_project,
        child_environment,
    )

    print(
        run_checked(
            pip_command + ["check"],
            clean_project,
            child_environment,
        ).strip()
    )

    # Confirm that the copied source files still match their fingerprints.
    run_checked(
        [str(clean_python), "download_data.py"],
        clean_project,
        child_environment,
    )

    print(
        run_checked(
            [str(clean_python), "tests/check_app.py"],
            clean_project,
            child_environment,
        ).strip()
    )

    report = json.loads(
        (clean_project / "VALIDATION.json").read_text(encoding="utf-8")
    )
    assert report["isolated_environment"], "The environment was not isolated."

    shutil.copy2(
        clean_project / "VALIDATION.json",
        PROJECT_DIR / "VALIDATION.json",
    )

    # Record transitive dependencies as well as the six direct dependencies.
    locked_versions = run_checked(
        pip_command + ["freeze"],
        clean_project,
        child_environment,
    )
    (PROJECT_DIR / "requirements-lock.txt").write_text(
        locked_versions,
        encoding="utf-8",
    )

readme_path = PROJECT_DIR / "README.md"
readme = readme_path.read_text(encoding="utf-8")
readme = readme.replace(
    "A full visual review and a clean-environment check remain to be completed.",
    "All four filter scenarios passed in a fresh virtual environment. "
    "A full visual review remains to be completed.",
)
readme = readme.replace(
    "- [ ] Complete the clean-environment check and visual review.",
    "- [x] Complete the clean-environment check.\n"
    "- [ ] Complete the full visual review.",
)

if "## Reproducing the validated environment" not in readme:
    readme += (
        "\n## Reproducing the validated environment\n\n"
        "Use `requirements-lock.txt` instead of `requirements.txt` to install "
        "the complete dependency versions recorded during validation.\n\n"
        "Run `python tests/check_app.py` inside the installed environment "
        "to repeat the functional checks. Results are saved in "
        "`VALIDATION.json`.\n"
    )

readme_path.write_text(readme, encoding="utf-8")

print("Clean-environment check passed.")
print("Saved: VALIDATION.json and requirements-lock.txt")
print("README validation status updated.")

Installing pinned dependencies in an isolated environment...
No broken requirements found.
2026-10-24 09:49:37.329 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
Passed: National
Passed: Guyane
Passed: Metropolitan France
Passed: Seine-et-Marne
Clean-environment check passed.
Saved: VALIDATION.json and requirements-lock.txt
README validation status updated.


In [40]:
import json

metadata = json.loads(
    (PROJECT_DIR / "project_metadata.json").read_text(encoding="utf-8")
)

storyboard = f"""# Storyboard — Access to General Practitioners in France

**Author:** {metadata["author"]}
**Student number / group:** {metadata["student_number"]} / {metadata["group"]}
**Course:** {metadata["course"]}
**Professor:** {metadata["professor"]}

## Purpose and audience

Help municipalities, healthcare centres and regional health authorities
identify geographical differences in potential GP access and decide where
further local investigation is needed.

## Motivation

I chose this topic because of my upcoming healthcare research internship
and my interest in territorial inequalities in access to care.

## Main question

How does potential access to general practitioners vary across France,
and where did it decline between 2023 and 2024?

## Opening hook

A national average can hide local gaps. In 2024, national potential access
was 3.26 annual consultations per age-standardised inhabitant, but access
varied substantially between and within departments.

## Story sequence

The figures below describe the national selection, excluding Mayotte.
Displayed results change when the user filters the dashboard.

| Stage | User question | Evidence and visual | Interaction | Intended takeaway |
|---|---|---|---|---|
| Overview | What is the overall situation? | Four KPI cards: APL, residents, department count and declining departments. | Select an area or departments in the sidebar. | National APL fell by 1.09%; 79 of 100 departments recorded a decrease. |
| Compare departments | Where is potential access lowest or highest? | Horizontal ranking with a national reference line. | Switch between lowest and highest APL; inspect hover details. | Guyane has the lowest departmental APL, around 1.41 compared with 3.26 nationally. |
| Explore geography | How are differences distributed geographically? | Department choropleth with a fixed national colour scale. | Change map focus between metropolitan France and overseas departments. | Geographical comparisons should include overseas territories and use a consistent scale. |
| Look within a department | What does the department average hide? | Population-weighted histogram and zero-APL indicators. | Select a department and hover over the distribution. | In Guyane, 16.3% of residents live in communes with zero APL for GPs aged 65 or younger. |
| Examine change | Which departments combine lower access with decline? | Scatter plot of 2024 APL against percentage change since 2023. | Hover over points and filter territories. | Fifty departments meet both conditions, containing 38.1% of the covered population. |
| Interpret and act | What should decision-makers investigate next? | Recommendations and expandable methods, limitations and policy context. | Open the relevant explanatory sections. | Investigate local appointment availability, staffing, transport and digital access before choosing interventions. |

## Interpretation safeguards

- APL measures potential access, not actual appointment waiting times.
- Weighted APL uses standardised population; resident shares use actual population.
- The national average is a benchmark, not an official shortage threshold.
- Zero APL does not establish an absence of all healthcare.
- Residents of a qualifying department do not all experience the same access.
- The population reference is 2022 for APL 2024 and 2021 for APL 2023.
- Teleconsultations are included in APL activity, but their separate contribution
  and platform effects cannot be measured from this workbook.
- The 2023-2024 observations cannot evaluate initiatives introduced later.
- Changes describe associations and trends; they do not establish causes.

## Planned demo — approximately 2 minutes 45 seconds

| Time | What to show | What to explain |
|---|---|---|
| 00:00–00:20 | Title and national overview. | Motivation, audience and the question behind the dashboard. |
| 00:20–00:45 | National KPI cards. | Meaning of APL, population reference and the 1.09% decline. |
| 00:45–01:15 | Ranking and map; switch to Guyane. | Department differences and the importance of including overseas territories. |
| 01:15–01:45 | Guyane distribution. | Why an average can conceal local gaps; explain the 16.3% figure carefully. |
| 01:45–02:15 | Return to national selection and show the scatter plot. | Explain the lower-left quadrant, 50 departments and the 38.1% population share. |
| 02:15–02:45 | Conclusions and limitations. | Suggest local investigation; clarify teleconsultations and the timing of policy initiatives. |

## Closing message

Use national comparisons to locate potential pressure, then examine local
differences before deciding what action is appropriate.
"""

(PROJECT_DIR / "STORYBOARD.md").write_text(
    storyboard.strip() + "\n",
    encoding="utf-8",
)

readme_path = PROJECT_DIR / "README.md"
readme = readme_path.read_text(encoding="utf-8")
readme = readme.replace(
    "- [ ] Add the final report and storyboard.",
    "- [ ] Add the final report.\n"
    "- [x] Add [STORYBOARD.md](STORYBOARD.md).",
)
readme_path.write_text(readme, encoding="utf-8")

print("Saved: STORYBOARD.md")
print("README checklist updated.")
print("Planned demo duration: approximately 2 minutes 45 seconds.")

Saved: STORYBOARD.md
README checklist updated.
Planned demo duration: approximately 2 minutes 45 seconds.


In [41]:
import json
import pandas as pd

data_dir = PROJECT_DIR / "data" / "processed"
metadata = json.loads(
    (PROJECT_DIR / "project_metadata.json").read_text(encoding="utf-8")
)

communes = pd.read_csv(
    data_dir / "apl_communes_2023_2024.csv",
    dtype={"commune_code": "string", "department_code": "string"},
)
departments = pd.read_csv(
    data_dir / "apl_departments_2024.csv",
    dtype={"department_code": "string"},
)
trends = pd.read_csv(
    data_dir / "apl_department_trends.csv",
    dtype={"department_code": "string"},
)
national = pd.read_csv(data_dir / "apl_national_trend.csv")

current = communes.loc[communes["apl_year"].eq(2024)]
previous = communes.loc[communes["apl_year"].eq(2023)]

apl_2024 = float(
    national.loc[national["apl_year"].eq(2024), "weighted_apl"].iloc[0]
)
apl_2023 = float(
    national.loc[national["apl_year"].eq(2023), "weighted_apl"].iloc[0]
)
change = 100 * (apl_2024 / apl_2023 - 1)
population = current["total_population"].sum()

declining = int(trends["absolute_change"].lt(0).sum())
both = trends.loc[
    trends["apl_2024"].lt(apl_2024)
    & trends["absolute_change"].lt(0)
]
both_population = departments.loc[
    departments["department_code"].isin(both["department_code"]),
    "total_population",
].sum()

guyane = current.loc[current["department_code"].eq("973")]
guyane_apl = (
    guyane["apl_65_or_younger"] * guyane["standardised_population"]
).sum() / guyane["standardised_population"].sum()

zero = guyane["apl_65_or_younger"].eq(0)
zero_residents = guyane.loc[zero, "total_population"].sum()
zero_share = 100 * zero_residents / guyane["total_population"].sum()
positive_all_ages = int((zero & guyane["apl_all_ages"].gt(0)).sum())

codes_2023 = set(previous["commune_code"])
codes_2024 = set(current["commune_code"])
duplicates = int(communes.duplicated(["apl_year", "commune_code"]).sum())

core_columns = [
    "commune_code", "commune_name", "department_code",
    "apl_65_or_younger", "standardised_population",
    "total_population", "apl_year", "population_year",
]
missing = int(communes[core_columns].isna().sum().sum())

highest_five = current.nlargest(5, "apl_65_or_younger")
without_highest = current.drop(highest_five.index)
sensitivity_apl = (
    without_highest["apl_65_or_younger"]
    * without_highest["standardised_population"]
).sum() / without_highest["standardised_population"].sum()

sections = [
    {
        "title": "1. Motivation and research question",
        "paragraphs": [
            "I chose this topic because of my upcoming healthcare research "
            "internship and my interest in territorial inequalities in access "
            "to care. A national figure can conceal very different local situations.",
            "The main question is: how does potential access to general "
            "practitioners vary across France, and where did it decline between "
            "2023 and 2024? The intended audience is municipalities, healthcare "
            "centres and regional health authorities.",
        ],
    },
    {
        "title": "2. Data and method",
        "paragraphs": [
            "The main dataset is DREES Localised Potential Accessibility (APL), "
            "referenced on data.gouv.fr. The analysis covers France excluding "
            "Mayotte and focuses on GPs aged 65 or younger. [1]",
            "APL describes annual accessible consultations, home visits and "
            "teleconsultations per age-standardised inhabitant. Department and "
            "national averages are weighted by standardised population. Actual "
            "population is used for resident counts and population shares. [2, 3]",
            f"The 2024 table contains {len(current):,} geographical units across "
            f"{len(departments)} departments and represents {population:,.0f} "
            "residents using the 2022 population reference. The 2023 APL "
            "observations use the 2021 population reference.",
        ],
    },
    {
        "title": "3. Preparation and validation",
        "paragraphs": [
            "Preparation preserves commune codes as strings, standardises column "
            "names, records reference years and joins department names. Numerical "
            "checks cover non-finite and negative values, age-subset consistency "
            "and population values.",
            f"The core prepared fields contain {missing} missing values and "
            f"{duplicates} duplicate commune-year pairs. Zero APL values are "
            "retained. Three zero-population observations in 2023 are retained "
            "and flagged; they contribute no population weight.",
            f"The geography differs between years: {len(codes_2023 - codes_2024)} "
            f"codes occur only in 2023 and {len(codes_2024 - codes_2023)} only in "
            "2024. Each year is therefore aggregated independently. Unmatched "
            "codes are not interpreted as practice openings or closures.",
            f"High values are retained. Removing the five highest 2024 APL "
            f"observations changes the national estimate from {apl_2024:.6f} "
            f"to {sensitivity_apl:.6f}. This sensitivity check indicates limited "
            "influence on the national average, without explaining those local values.",
        ],
    },
    {
        "title": "4. National and departmental findings",
        "paragraphs": [
            f"National weighted APL decreased from {apl_2023:.3f} in 2023 to "
            f"{apl_2024:.3f} in 2024, a change of {change:+.2f}%. "
            f"{declining} of {len(trends)} departments recorded a decrease.",
            f"{len(both)} departments combined below-national APL in 2024 with "
            f"a decline since 2023. They contained {both_population:,.0f} "
            f"residents, or {100 * both_population / population:.1f}% of the "
            "covered population using the 2022 reference.",
            "These results identify territories for closer investigation. They "
            "do not mean every resident of those departments experiences low "
            "access, and the national average is not an official shortage threshold.",
        ],
    },
    {
        "title": "5. Local case study: Guyane",
        "paragraphs": [
            f"Guyane's departmental APL was {guyane_apl:.3f}, compared with "
            f"{apl_2024:.3f} nationally. Among its {len(guyane)} communes, "
            f"{int(zero.sum())} had zero APL for GPs aged 65 or younger. These "
            f"communes contained {zero_residents:,.0f} residents "
            f"({zero_share:.1f}% of the department population).",
            f"{positive_all_ages} of these communes have positive APL when GPs "
            "of all ages are included. Zero values must therefore be interpreted "
            "within the indicator's professional and age scope; they do not "
            "establish an absence of all healthcare.",
        ],
    },
    {
        "title": "6. Teleconsultations and policy context",
        "paragraphs": [
            "Teleconsultations are included in APL activity. However, this workbook "
            "does not separate remote care from in-person care, identify platforms "
            "such as Doctolib, or quantify how far remote care offsets isolation. [3]",
            "A separate DREES study of adults in metropolitan France found that "
            "21% of large urban-centre residents used teleconsultation in 2024, "
            "compared with around 10% in rural areas. Among teleconsultation users, "
            "43% cited long waits for in-person appointments. This study includes "
            "GPs and specialists and does not support assuming that the most "
            "isolated residents use remote care most. [4]",
            "Policy context checked on 23 October 2026 includes Un médecin près "
            "de chez vous, operating since September 2025 in 153 priority "
            "territories with voluntary participation in its pilot implementation, "
            "and the France Santé network, whose June 2026 circular sets a target "
            "of 5,000 labelled structures by 2027. These are programme descriptions "
            "and targets, not effects demonstrated here. The 2023–2024 observations "
            "predate these initiatives and cannot evaluate their effectiveness. [5, 6]",
        ],
    },
    {
        "title": "7. Dashboard design and reproducibility",
        "paragraphs": [
            "The dashboard follows a sequence from national context to department "
            "comparisons, geography, local distributions, recent change and "
            "implications. Four interactive charts provide hover information. "
            "Sidebar filters update the KPIs and analytical selection; the map "
            "focus controls the displayed geography.",
            "The application uses modular Python files, cached loading, prepared "
            "CSV tables and a fixed map colour scale. Source fingerprints, pinned "
            "requirements and a complete dependency lock are included. Four filter "
            "scenarios passed in a fresh virtual environment. Full visual review "
            "remains separate from these execution checks.",
        ],
    },
    {
        "title": "8. Limitations and implications",
        "paragraphs": [
            "APL measures potential access rather than actual waiting times, "
            "acceptance of new patients, affordability or care quality. Department "
            "averages hide local differences. Changes are descriptive and do not "
            "establish causes. Population standardisation does not fully account "
            "for evolving needs caused by national population ageing between years. [2]",
            "Local decision-makers can use these results to structure discussions "
            "about staffing, transport, appointment availability and digital access. "
            "The next step is to combine the geographical signals with local service "
            "information and later APL releases before evaluating interventions.",
        ],
    },
]

references = [
    {"id": 1, "title": "DREES APL dataset", "url": apl_source},
    {
        "id": 2,
        "title": "DREES methodological notice",
        "url": (
            "https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/"
            "datasets/530_l-accessibilite-potentielle-localisee-apl/"
            "attachments/apl_lisez_moi_pdf"
        ),
    },
    {"id": 3, "title": "DREES: access to primary care in 2024", "url": apl_release},
    {"id": 4, "title": "DREES: teleconsultations, ER 1366", "url": tele_source},
    {"id": 5, "title": "Ministry: Un médecin près de chez vous", "url": policy_source},
    {"id": 6, "title": "France Santé circular, 16 June 2026", "url": france_sante_source},
    {
        "id": 7,
        "title": "Etalab/IGN department boundaries — ODbL",
        "url": "https://www.data.gouv.fr/datasets/contours-administratifs",
    },
    {
        "id": 8,
        "title": "French geographic API: department names",
        "url": "https://geo.api.gouv.fr/departements",
    },
]

report_content = {
    "title": "Access to General Practitioners in France",
    "metadata": metadata,
    "sections": sections,
    "references": references,
}

(PROJECT_DIR / "REPORT_CONTENT.json").write_text(
    json.dumps(report_content, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

author_details = (
    f"**Author:** {metadata['author']}  \n"
    f"**Student number / group:** {metadata['student_number']} / {metadata['group']}  \n"
    f"**Institution:** {metadata['institution']}  \n"
    f"**Programme / year:** {metadata['programme']} / {metadata['academic_year']}  \n"
    f"**Course:** {metadata['course']}  \n"
    f"**Professor:** {metadata['professor']}  \n"
    f"**Email:** {metadata['email']}  \n"
    f"**LinkedIn:** {metadata['linkedin']}"
)

report_markdown = (
    f"# {report_content['title']}\n\n{author_details}\n\n"
    + "\n\n".join(
        f"## {section['title']}\n\n" + "\n\n".join(section["paragraphs"])
        for section in sections
    )
    + "\n\n## References\n\n"
    + "\n".join(
        f"- [{ref['id']}] [{ref['title']}]({ref['url']})"
        for ref in references
    )
    + "\n"
)

(PROJECT_DIR / "REPORT.md").write_text(
    report_markdown,
    encoding="utf-8",
)

print("Saved: REPORT.md")
print("Saved: REPORT_CONTENT.json")
print("Report figures calculated from the exported tables.")

Saved: REPORT.md
Saved: REPORT_CONTENT.json
Report figures calculated from the exported tables.


In [42]:
import base64
import json
import re
from html import escape

import pandas as pd
from google.colab import files

content = json.loads(
    (PROJECT_DIR / "REPORT_CONTENT.json").read_text(encoding="utf-8")
)
details = json.loads(
    (PROJECT_DIR / "project_metadata.json").read_text(encoding="utf-8")
)

logo_base64 = base64.b64encode(
    (PROJECT_DIR / "assets" / "efrei_logo.png").read_bytes()
).decode("ascii")

national_data = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "apl_national_trend.csv"
)
department_data = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "apl_department_trends.csv"
)

apl_current = float(
    national_data.loc[
        national_data["apl_year"].eq(2024), "weighted_apl"
    ].iloc[0]
)
apl_previous = float(
    national_data.loc[
        national_data["apl_year"].eq(2023), "weighted_apl"
    ].iloc[0]
)
annual_change = 100 * (apl_current / apl_previous - 1)
declining_count = int(department_data["absolute_change"].lt(0).sum())


def format_paragraph(text):
    safe_text = escape(text)

    def link_references(match):
        numbers = re.findall(r"\d+", match.group(1))
        links = [
            f'<a href="#ref-{number}">{number}</a>'
            for number in numbers
        ]
        return "[" + ", ".join(links) + "]"

    return re.sub(
        r"\[(\d+(?:,\s*\d+)*)\]",
        link_references,
        safe_text,
    )


identity_fields = [
    ("Student number", details["student_number"]),
    ("Group", details["group"]),
    ("Institution", details["institution"]),
    ("Programme", details["programme"]),
    ("Academic year", details["academic_year"]),
    ("Course", details["course"]),
    ("Professor", details["professor"]),
]

identity_html = "".join(
    f'<div class="detail"><span>{escape(label)}</span>'
    f'<strong>{escape(str(value))}</strong></div>'
    for label, value in identity_fields
)

sections_html = "".join(
    f'<section><h2>{escape(section["title"])}</h2>'
    + "".join(
        f"<p>{format_paragraph(paragraph)}</p>"
        for paragraph in section["paragraphs"]
    )
    + "</section>"
    for section in content["sections"]
)

references_html = "".join(
    f'<li id="ref-{reference["id"]}">'
    f'<a href="{escape(reference["url"], quote=True)}">'
    f'{escape(reference["title"])}</a></li>'
    for reference in content["references"]
)

submission_links = "".join(
    f'<p><a href="{escape(details[key], quote=True)}">{label}</a></p>'
    for key, label in [
        ("app_url", "Open the dashboard"),
        ("repository_url", "Project repository"),
        ("demo_video_url", "Demo video"),
    ]
    if details.get(key)
)

styles = """
* { box-sizing: border-box; }
body {
    margin: 0; background: #eef3f8; color: #202c38;
    font-family: Arial, Helvetica, sans-serif; line-height: 1.6;
}
.toolbar {
    max-width: 1120px; margin: 18px auto; padding: 0 16px;
}
button {
    border: 0; border-radius: 6px; padding: 11px 18px;
    background: #086da8; color: white; cursor: pointer; font-size: 14px;
}
.sheet {
    max-width: 1120px; margin: 0 auto 36px; padding: 36px;
    display: grid; grid-template-columns: 210px minmax(0, 1fr);
    gap: 34px; background: white;
}
.identity {
    border-right: 1px solid #cbd7e2; padding-right: 22px;
    font-size: 12px; overflow-wrap: anywhere;
}
.identity img { width: 165px; max-width: 100%; height: auto; }
.identity h2 { font-size: 19px; margin: 22px 0 18px; color: #075d91; }
.detail { margin-bottom: 13px; }
.detail span { display: block; color: #576879; font-size: 11px; }
.detail strong { display: block; font-weight: 600; }
.identity a { display: block; margin-top: 12px; }
.eyebrow { color: #536d82; font-size: 12px; margin: 0 0 10px; }
h1 { color: #123c59; font-size: 32px; line-height: 1.2; margin: 0 0 12px; }
.subtitle { color: #536473; font-size: 14px; margin-bottom: 24px; }
.kpis {
    display: grid; grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 12px; margin: 22px 0 10px;
}
.kpi { background: #edf5fa; border-top: 3px solid #0875b1; padding: 14px; }
.kpi strong { display: block; color: #075d91; font-size: 25px; }
.kpi span { font-size: 11px; color: #354e61; }
.units { font-size: 11px; color: #536473; margin-bottom: 25px; }
h2 { color: #075d91; font-size: 19px; line-height: 1.3; margin: 28px 0 12px; }
p { margin: 0 0 12px; font-size: 14px; }
a { color: #075d91; text-decoration: underline; }
.references { padding-left: 22px; font-size: 12px; }
.references li { margin-bottom: 9px; overflow-wrap: anywhere; }
.attribution {
    margin-top: 28px; border-top: 1px solid #cbd7e2;
    padding-top: 12px; color: #536473; font-size: 11px;
}
@media screen and (max-width: 700px) {
    .sheet { display: block; padding: 22px; }
    .identity {
        border-right: 0; border-bottom: 1px solid #cbd7e2;
        padding: 0 0 20px; margin-bottom: 25px;
    }
    h1 { font-size: 27px; }
}
@page { size: A4; margin: 15mm 14mm; }
@media print {
    body { background: white; }
    .toolbar { display: none; }
    .sheet { display: block; margin: 0; padding: 0; max-width: none; }
    .identity {
        position: fixed; top: 0; left: 0; width: 39mm;
        padding-right: 4mm; font-size: 8pt;
    }
    .identity img { width: 35mm; }
    .identity h2 { font-size: 12pt; margin: 6mm 0 5mm; }
    .detail { margin-bottom: 3mm; }
    .detail span { font-size: 7.5pt; }
    main { margin-left: 47mm; }
    h1 { font-size: 22pt; }
    h2 { font-size: 12pt; break-after: avoid; margin-top: 6mm; }
    p { font-size: 9.5pt; orphans: 3; widows: 3; }
    .eyebrow, .subtitle { font-size: 9pt; }
    .kpis { gap: 2mm; break-inside: avoid; }
    .kpi { padding: 3mm; print-color-adjust: exact; }
    .kpi strong { font-size: 17pt; }
    .kpi span, .units, .attribution, .references { font-size: 8pt; }
    .references li { break-inside: avoid; }
}
"""

report_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{escape(content["title"])} — {escape(details["author"])}</title>
<style>{styles}</style>
</head>
<body>
<div class="toolbar">
    <button onclick="window.print()">Print / Save as PDF</button>
</div>
<div class="sheet">
    <aside class="identity">
        <img src="data:image/png;base64,{logo_base64}" alt="EFREI logo">
        <h2>{escape(details["author"])}</h2>
        {identity_html}
        <a href="mailto:{escape(details["email"], quote=True)}">
            {escape(details["email"])}
        </a>
        <a href="{escape(details["linkedin"], quote=True)}">LinkedIn profile</a>
        {submission_links}
    </aside>
    <main>
        <p class="eyebrow">DATA ANALYSIS &amp; VISUALIZATION · STUDENT REPORT</p>
        <h1>{escape(content["title"])}</h1>
        <p class="subtitle">
            DREES APL · 2024 snapshot with a 2023 comparison
        </p>
        <div class="kpis">
            <div class="kpi">
                <strong>{apl_current:.2f}</strong>
                <span>National APL, 2024</span>
            </div>
            <div class="kpi">
                <strong>{annual_change:+.2f}%</strong>
                <span>Change since 2023</span>
            </div>
            <div class="kpi">
                <strong>{declining_count} / {len(department_data)}</strong>
                <span>Declining departments</span>
            </div>
        </div>
        <p class="units">
            France excluding Mayotte · GPs aged 65 or younger.
            APL: annual accessible consultations per age-standardised inhabitant.
        </p>
        {sections_html}
        <h2>References</h2>
        <ol class="references">{references_html}</ol>
        <p class="attribution">
            DREES APL data: Open Licence 2.0.
            Map boundaries: Etalab/IGN, ODbL [7].
            Department names: French geographic API [8].
            EFREI branding is separate from the data licences.
        </p>
    </main>
</div>
</body>
</html>
"""

report_path = PROJECT_DIR / "REPORT.html"
report_path.write_text(report_html, encoding="utf-8")

print("Saved: REPORT.html")
print("Open the downloaded file to review the layout.")
files.download(str(report_path))

Saved: REPORT.html
Open the downloaded file to review the layout.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
from google.colab import files
from pathlib import Path

PROJECT_DIR = Path("/content/apl_dashboard")

uploaded = files.upload()

assert len(uploaded) == 1, "Select only REPORT.pdf."
filename, pdf_bytes = next(iter(uploaded.items()))

assert filename.lower().endswith(".pdf"), "Select a PDF file."
assert pdf_bytes.startswith(b"%PDF-"), "The selected file is not a PDF."

pdf_path = PROJECT_DIR / "REPORT.pdf"
pdf_path.write_bytes(pdf_bytes)

print("Saved: REPORT.pdf")
print("PDF added to the project. Page-layout review pending.")

Saving Access to General Practitioners in France - Yanis AMARA.pdf to Access to General Practitioners in France - Yanis AMARA (2).pdf
Saved: REPORT.pdf
PDF added to the project. Page-layout review pending.
